# Stage B / NB 07 — Zero-shot VLM agents and token-probability scoring (E0a, E0b, E0c)

Protocol reference: Section 6.2 experiments **E0a/E0b/E0c**; Section **7.2** (the enabling
fix); research questions **RQ1**, **RQ2**. Addresses referee 1.3 (no ROC/AUC/PR curves) and
referee 2d (no AUROC/F1) at the root.

## This notebook carries the fix the whole revision depends on

The rejected submission reported accuracy and specificity but no AUROC, no PR curve, and no
confidence intervals. That was not an oversight in the analysis — it was a consequence of the
architecture. A generative model emitting `{"covid_positive":"Yes"}` produces a *string*. There
is no probability, so ROC, precision–recall, DeLong, calibration, and threshold analysis are
all unavailable.

Protocol Section 7.2 fixes this by scoring, not just generating. Two passes per image:

1. **Free generation** (`do_sample=False`) → the parsed decision, JSON validity, latency. This
   is what the model would actually output in deployment.
2. **Constrained scoring** → teacher-force each candidate completion and compute its exact
   sequence log-likelihood. For COVID that is two candidates (`Yes` / `No`), giving
   `p(COVID) = softmax over the two sequence log-probs`. For mRALE it reads the token
   distribution at each numeric field position, giving a soft expectation and a variance.

Sequence-likelihood scoring rather than "read the first answer token" is deliberate: it is
model-agnostic, handles multi-token and whitespace variants, and needs no assumption about
tokenisation. The cost is one extra short forward pass per candidate.

**Validation requirement.** `argmax` over the scored candidates must reproduce the free-generation
decision on ≥ 99.5% of **decisive** cases (default absolute log-likelihood margin ≥ 0.5 nat).
A failing arm is quarantined from ROC/PR/DeLong/calibration; the gate blocks only if every arm
fails, which indicates a systematic scoring problem. Overall agreement and the near-tie fraction
are still reported so the decisive-case rule is auditable.

## The three models

| arm | model | question |
| --- | --- | --- |
| E0a | google/medgemma-1.5-4b-it | medical multimodal pretraining, zero-shot |
| E0b | Qwen/Qwen3.5-4B | matched-scale general-purpose control → **RQ1** |
| E0c | nvidia/NV-Reason-CXR-3B | reasoning-tuned CXR model → **RQ2** |

E0a vs E0b at matched 4B scale is the medical-specialization test. E0c decides whether the
optional LoRA stretch arm (E5-S) is worth running at all.

## Also fixed here: E6-Q

The tested fold-0 predictions emit the JSON object and then repeat it, with a stray
`<unused94>model` token. Parsing recovers the first object so the old metrics were valid, but
token accounting, latency, and any self-consistency arm would be corrupted. This notebook adds
stop criteria and reports a `duplicate_object_rate` so the fix is measured, not assumed.

## Outputs (under `stage_B/nb07_zeroshot/`)
- `predictions_zeroshot.jsonl` (checkpointed per image, resumable)
- `token_probability_scores.parquet`
- `arm_summary.csv`, `score_validation.csv`, `nvreason_traces.jsonl`
- `run_config.json`, `gate_nb07.json`

## 1. Imports, seeds, and the Stage A path contract

In [ ]:
import gc
import json
import math
import os
import random
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions. Table 2 is only a valid comparison if every arm uses these.
_METRICS_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd() / "stage_B",
                   Path.cwd().parent / "stage_B"]
for _candidate in _METRICS_SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "cxr_metrics.py not found. It must sit beside the Stage B notebooks; every arm in "
        f"Table 2 depends on its metric definitions. Searched: {_METRICS_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---- Stage A path contract -------------------------------------------------------------
FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
    Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Path contract:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # matches the tested LoRA notebooks

print("Stage B output root:", STAGE_B_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

## 2. Configuration

In [ ]:
import re

NB07_DIR = STAGE_B_DIR / "nb07_zeroshot"
NB07_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ARMS = OrderedDict([
    ("E0a_medgemma", {
        "model_id": "google/medgemma-1.5-4b-it",
        "loader": "image_text_to_text",
        "agent": "A2_medgemma_zeroshot",
        "capture_traces": False,
    }),
    ("E0b_qwen35", {
        "model_id": "Qwen/Qwen3.5-4B",
        "loader": "multimodal_lm",
        "agent": "A3_qwen_zeroshot",
        "capture_traces": False,
        # Qwen3.5 thinks by default; direct JSON supervision and scoring must not include a
        # <think> trace, matching the tested LoRA notebook's DISABLE_THINKING=True.
        "disable_thinking": True,
    }),
    ("E0c_nvreason", {
        "model_id": "nvidia/NV-Reason-CXR-3B",
        "loader": "auto",
        "agent": "A4_nvreason_zeroshot",
        "capture_traces": True,          # its reasoning traces feed E10c grounding analysis
    }),
])

RUN_ARMS = list(MODEL_ARMS)
TASKS = ["covid_classification", "mrale_prediction"]

MAX_NEW_TOKENS = {"covid_classification": 48, "mrale_prediction": 256}
TRACE_MAX_NEW_TOKENS = 768               # NV-Reason emits a long chain of thought

RUN_TOKEN_SCORING = True
SCORE_MRALE_FIELDS = True
COVID_CANDIDATES = ["Yes", "No"]

INCLUDE_EXTERNAL = True
MAX_IMAGES_PER_ARM = None                # set 16 for a wiring check, then None

# E6-Q: stop as soon as the first JSON object closes, so the model cannot repeat it.
USE_JSON_STOP_CRITERIA = True

# ---- Resume / skip control ---------------------------------------------------------------
# Sections 7-9 are expensive. Each writes a completion marker with the row count and a hash
# of its inputs; on a re-run, a section whose inputs are unchanged is SKIPPED and its saved
# artifacts are reloaded. Set the corresponding FORCE_* flag to recompute deliberately.
FORCE_REGENERATE = False        # section 7: generation + first-pass scoring
FORCE_RESCORE = False           # section 8b: prefix-aligned rescoring
FORCE_RECOMPUTE_METRICS = False # sections 8-9: validation, metrics, integrity

# ---- Protocol 7.2 enforcement -------------------------------------------------------------
# An arm whose scored decision does not reproduce its generated decision has an UNUSABLE
# continuous score. Quarantining it (rather than aborting) marks that arm score_usable=False
# for NB 17/18 and lets the other arms' results be saved. The gate still fails if EVERY arm
# fails, because then there is no AUROC anywhere and referee 1.3 is unanswered.
SCORE_AGREEMENT_THRESHOLD = 0.995
QUARANTINE_FAILING_SCORE_ARMS = True

# The 0.995 threshold implicitly assumes the model has a PREFERENCE on every image. True for an
# adapted model; false for a near-chance zero-shot one. When |logL_yes - logL_no| is tiny the
# model is genuinely undecided, and greedy free-generation and likelihood ranking can disagree
# on such a tie without either being wrong -- so counting ties as scoring failures would reject
# a faithful score.
#
# Agreement is reported twice: over all scored cases, and over DECISIVE cases only. The gate
# uses the decisive figure, because the claim that matters is that the score reflects the
# model's preference WHERE IT HAS ONE. The near-tie fraction is reported alongside so the
# exclusion is visible rather than hidden -- silently dropping cases from a validation
# statistic is exactly what referee 1.3 is asking us not to do.
AGREEMENT_DECISIVE_MARGIN = 0.5      # nats
GATE_ON_DECISIVE_AGREEMENT = True

# ---- Image-binding preflight --------------------------------------------------------------
# A model that never receives the image produces confident-looking text with zero valid JSON
# ("Please provide a frontal chest X-ray image."). Twenty hours of that is indistinguishable
# from a weak model until you read the raw output. RUN_IMAGE_PREFLIGHT catches it in seconds
# per arm, before any generation.
RUN_IMAGE_PREFLIGHT = True
IMAGE_PLACEHOLDER_TOKENS = (
    "<image>", "<|image_pad|>", "<|vision_start|>", "<start_of_image>",
    "<image_soft_token>", "<img>", "[IMG]",
)
# Per-arm message schema, filled in automatically by the schema probe in section 6c.
# Leave empty to auto-detect; set explicitly to pin a model to a known-good convention.
ARM_MESSAGE_SCHEMA = {}

IMAGE_ABSENT_PHRASES = (
    "provide a", "provide the", "no image", "cannot see", "can't see", "unable to view",
    "upload", "i don't see", "there is no image", "please share",
)

# Section 8b: re-score these arms using candidates built from the model's OWN emitted format
# instead of the canonical JSON string. Leave empty to skip. Cached generations are reused, so
# this costs only the scoring forward passes (~1/3 of a full arm pass).
RESCORE_ARMS = []

# Arms whose cached predictions are DISCARDED so they regenerate from scratch. Use this after
# changing an arm's prompt or decoding settings. Re-appended rows supersede the old ones
# (load_jsonl_by_key keeps the last occurrence per key), so the JSONL needs no editing.
RESET_ARMS = []

# Arm-level prompt suffixes, appended to the shared user prompt for that arm only.
#
# Needed for chain-of-thought models: NV-Reason-CXR-3B narrates and may never emit bare JSON
# under the shared template, which produces 0.000 mRALE coverage. A schema reminder fixes it.
#
# WARNING: this is a DEVIATION that must be disclosed. If NV-Reason receives different
# instructions from MedGemma and Qwen, RQ2 is no longer a clean matched comparison, and the
# manuscript has to say so rather than presenting the arms as identically prompted.
ARM_PROMPT_SUFFIX = {
    # "E0c_nvreason": (
    #     " Think silently. Your entire reply must be one JSON object and nothing else: no "
    #     "explanation, no markdown fence, no text before or after the closing brace."
    # ),
}

# An arm producing zero parseable mRALE outputs is a FORMAT BUG, not a weak result, and is
# treated as a gate failure rather than a warning.
MIN_ACCEPTABLE_MRALE_COVERAGE = 0.0
# Zero coverage is a bug, but blocking the whole notebook over one arm discards the others'
# work. Consistent with the score policy: quarantine the arm (mrale_usable=False), warn loudly,
# and fail only if EVERY arm has zero coverage.
QUARANTINE_ZERO_COVERAGE_ARMS = True

print("Arms:", RUN_ARMS)
print("Token scoring:", RUN_TOKEN_SCORING)
print("Output:", NB07_DIR)
for arm, spec in MODEL_ARMS.items():
    print(f"  {arm}: {spec['model_id']} revision={MODEL_REVISIONS.get(spec['model_id'])}")

## 2b. Resume control

Each expensive section writes a marker recording a fingerprint of its inputs. On a re-run, a section whose inputs are unchanged is skipped and its artifacts are reloaded. The generation loop in section 7 is additionally checkpointed per `(arm, image, task)`, so even a partial section can resume mid-arm.


In [ ]:
# Stage completion markers, so a 20-hour section is never silently repeated.
import hashlib

STATUS_PATH = NB07_DIR / "stage_status.json"


def _load_status():
    if STATUS_PATH.is_file():
        try:
            return json.loads(STATUS_PATH.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}


def _save_status(status):
    cm.write_json(STATUS_PATH, status)


def fingerprint(*parts):
    digest = hashlib.sha256()
    for part in parts:
        digest.update(str(part).encode("utf-8"))
    return digest.hexdigest()[:16]


def stage_complete(name, expected_fingerprint, required_files=()):
    """True when this stage already ran against identical inputs and its outputs exist."""
    entry = _load_status().get(name)
    if not entry or entry.get("fingerprint") != expected_fingerprint:
        return False
    for path in required_files:
        if not Path(path).is_file():
            print(f"  [{name}] marker present but {Path(path).name} is missing; recomputing.")
            return False
    return True


def mark_stage(name, expected_fingerprint, detail=None):
    status = _load_status()
    status[name] = {
        "fingerprint": expected_fingerprint,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "detail": detail or {},
    }
    _save_status(status)
    print(f"  [{name}] marked complete ({expected_fingerprint})")


status_snapshot = _load_status()
if status_snapshot:
    print("Previously completed stages:")
    for name, entry in status_snapshot.items():
        detail = entry.get("detail", {})
        print(f"  {name:<24} {entry.get('completed_utc', '?')[:19]}  {detail}")
else:
    print("No previous stage markers; this is a fresh run.")


## 3. Prompts

Harvested from NB 02's `prompt_templates.json`, which itself harvested them from the tested
fold files. Zero-shot models must see exactly the instructions the LoRA arms were trained on,
or E0a-vs-NB08 stops being a clean zero-shot-versus-adapted comparison and becomes a prompt
comparison as well.

In [ ]:
templates_path = NB02_DIR / "prompt_templates.json"
if not templates_path.is_file():
    raise FileNotFoundError(
        f"{templates_path} not found. Run Stage A NB 02 first: zero-shot arms must use the "
        "same prompts as the LoRA arms or the comparison is confounded."
    )
templates = json.loads(templates_path.read_text(encoding="utf-8"))["templates"]
for task in TASKS:
    if task not in templates:
        raise KeyError(f"No prompt template for {task} in {templates_path}")
    print(f"--- {task}")
    print("  system:", templates[task]["system"][:130], "...")
    print("  user  :", templates[task]["user"][:130], "...")


def multimodal_messages(system_prompt, user_prompt):
    return [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_prompt},
        ]},
    ]


# Canonical target JSON, used to build scoring candidates. The key order matches the
# ground-truth strings NB 02 regenerated, so a teacher-forced candidate is token-identical
# to what a correct model would emit.
def covid_candidate_json(value):
    return json.dumps({"covid_positive": value}, separators=(",", ":"))


MRALE_KEY_ORDER = [
    "extent_right", "density_right", "extent_left", "density_left",
    "extent_right_numerical", "density_right_numerical",
    "extent_left_numerical", "density_left_numerical", "mRALE Score",
]
print()
print("Scoring candidates for COVID:", [covid_candidate_json(v) for v in COVID_CANDIDATES])

## 4. Model loading, generation, and JSON stop criteria

In [ ]:
from PIL import Image
from transformers import AutoProcessor, StoppingCriteria, StoppingCriteriaList

Image.MAX_IMAGE_PIXELS = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32


class BalancedJsonStop(StoppingCriteria):
    """
    E6-Q fix. Stops as soon as the first top-level JSON object closes.

    The tested notebooks let the model keep generating after the object closed, so it emitted
    the same object several times plus stray control tokens. Parsing recovered the first copy,
    so the metrics were valid, but token counts, latency, and any self-consistency aggregation
    were measuring repetition rather than the answer.
    """

    def __init__(self, tokenizer, prompt_length):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        text = self.tokenizer.decode(
            input_ids[0, self.prompt_length:], skip_special_tokens=True)
        start = text.find("{")
        if start < 0:
            return False
        depth, in_string, escape = 0, False, False
        for character in text[start:]:
            if escape:
                escape = False
                continue
            if character == "\\":
                escape = True
                continue
            if character == '"':
                in_string = not in_string
                continue
            if in_string:
                continue
            if character == "{":
                depth += 1
            elif character == "}":
                depth -= 1
                if depth == 0:
                    return True
        return False


def load_vlm(spec):
    model_id = spec["model_id"]
    revision = MODEL_REVISIONS.get(model_id)
    kwargs = {"trust_remote_code": True}
    if revision:
        kwargs["revision"] = revision

    processor = AutoProcessor.from_pretrained(model_id, **kwargs)
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"

    loader = spec["loader"]
    if loader == "image_text_to_text":
        from transformers import AutoModelForImageTextToText as ModelClass
    elif loader == "multimodal_lm":
        from transformers import AutoModelForMultimodalLM as ModelClass
    else:
        try:
            from transformers import AutoModelForImageTextToText as ModelClass
        except Exception:
            from transformers import AutoModelForVision2Seq as ModelClass

    model = ModelClass.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map="auto", low_cpu_mem_usage=True, **kwargs)
    model.config.use_cache = True
    model.eval()
    return model, processor, revision


def model_device(model):
    for parameter in model.parameters():
        if parameter.device.type not in {"meta", "cpu"}:
            return parameter.device
    return device


def _apply_template(processor, spec, messages):
    kwargs = {"add_generation_prompt": True, "tokenize": False}
    if spec.get("disable_thinking"):
        # Qwen3.5 exposes this through the chat template; ignored by templates without it.
        kwargs["enable_thinking"] = False
    try:
        return processor.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop("enable_thinking", None)
        return processor.apply_chat_template(messages, **kwargs)


def has_image_placeholder(text):
    return any(token in text for token in IMAGE_PLACEHOLDER_TOKENS)


def vision_token_string(processor, model=None):
    """
    The literal image placeholder this model expects, discovered from the processor/config.

    Used by the `manual_vision_tokens` schema, which is the guaranteed fallback: if a chat
    template refuses to emit a vision token for any message form, we insert the token
    ourselves rather than letting the model run blind.
    """
    tokenizer = processor.tokenizer
    for attribute in ["image_token", "boi_token"]:
        token = getattr(processor, attribute, None) or getattr(tokenizer, attribute, None)
        if isinstance(token, str) and token:
            # Qwen2.5-VL wraps the pad token in vision start/end markers.
            vocab = tokenizer.get_vocab()
            if "<|vision_start|>" in vocab and "<|vision_end|>" in vocab:
                return f"<|vision_start|>{token}<|vision_end|>"
            return token
    vocab = tokenizer.get_vocab()
    if "<|image_pad|>" in vocab:
        return "<|vision_start|><|image_pad|><|vision_end|>"
    for candidate in ["<start_of_image>", "<image_soft_token>", "<image>", "<img>"]:
        if candidate in vocab:
            return candidate
    if model is not None:
        for attribute in ["image_token_id", "image_token_index"]:
            token_id = getattr(getattr(model, "config", None), attribute, None)
            if isinstance(token_id, int):
                return tokenizer.decode([token_id])
    return None


MESSAGE_SCHEMAS = [
    "bare", "valued", "file_uri", "system_as_string",
    "system_folded_into_user", "manual_vision_tokens",
]


def build_messages(schema, system_prompt, user_prompt, image_path, processor, model=None):
    """Return (messages, note). Raises ValueError if the schema cannot be constructed."""
    path = str(image_path) if image_path is not None else None
    if schema == "bare":
        return multimodal_messages(system_prompt, user_prompt), "{'type': 'image'} with no value"
    if schema == "valued":
        return ([{"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                 {"role": "user", "content": [{"type": "image", "image": path},
                                              {"type": "text", "text": user_prompt}]}],
                "image path supplied inline")
    if schema == "file_uri":
        return ([{"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                 {"role": "user", "content": [{"type": "image", "image": f"file://{path}"},
                                              {"type": "text", "text": user_prompt}]}],
                "file:// URI, the Qwen2.5-VL documented form")
    if schema == "system_as_string":
        return ([{"role": "system", "content": system_prompt},
                 {"role": "user", "content": [{"type": "image", "image": path},
                                              {"type": "text", "text": user_prompt}]}],
                "plain-string system turn")
    if schema == "system_folded_into_user":
        return ([{"role": "user", "content": [
                    {"type": "image", "image": path},
                    {"type": "text", "text": f"{system_prompt}\n\n{user_prompt}"}]}],
                "no system turn; instructions folded into the user turn")
    if schema == "manual_vision_tokens":
        token = vision_token_string(processor, model)
        if not token:
            raise ValueError("no vision token could be discovered for this model")
        return ([{"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                 {"role": "user", "content": [
                     {"type": "text", "text": f"{token}\n{user_prompt}"}]}],
                f"vision token {token!r} injected into the user text")
    raise ValueError(f"unknown schema {schema}")


def render_prompt(processor, spec, system_prompt, user_prompt, image_path=None, model=None):
    """
    Render the chat prompt, ENSURING it contains the model's image placeholder.

    `{"type": "image"}` with no value is enough for MedGemma and Qwen3.5, but some templates
    (notably Qwen2.5-VL derivatives such as NV-Reason-CXR-3B) emit nothing for it. The prompt
    then contains no vision token, the image is never attended to, and the model politely asks
    for an image while every JSON parse fails. Variants are tried until one produces a
    placeholder, and which variant was used is recorded per arm.
    """
    # A pinned schema (set explicitly, or chosen by the section 6c probe) wins outright.
    pinned = ARM_MESSAGE_SCHEMA.get(spec["model_id"]) or ARM_MESSAGE_SCHEMA.get(
        spec.get("_arm", ""))
    order = ([pinned] + [name for name in MESSAGE_SCHEMAS if name != pinned]
             if pinned else list(MESSAGE_SCHEMAS))

    first_text, first_name, errors = None, None, {}
    for name in order:
        try:
            messages, _ = build_messages(
                name, system_prompt, user_prompt, image_path, processor, model)
            text = _apply_template(processor, spec, messages)
        except Exception as exc:
            errors[name] = f"{type(exc).__name__}: {exc}"
            continue
        if first_text is None:
            first_text, first_name = text, name
        if has_image_placeholder(text):
            IMAGE_TEMPLATE_VARIANT[spec["model_id"]] = name
            return text
    IMAGE_TEMPLATE_VARIANT[spec["model_id"]] = (
        f"NO_PLACEHOLDER(first={first_name}, errors={errors})"
        if first_text is not None else f"ALL_FAILED({errors})")
    if first_text is None:
        raise RuntimeError(f"Could not render a prompt for {spec['model_id']}: {errors}")
    return first_text


IMAGE_TEMPLATE_VARIANT = {}


def prepare_inputs(processor, prompt_text, image_path, target_device):
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
        inputs = processor(text=prompt_text, images=image, return_tensors="pt")
    return {
        key: (value.to(device=target_device, dtype=DTYPE)
              if value.is_floating_point() else value.to(target_device))
        for key, value in inputs.items()
    }


@torch.inference_mode()
def generate_text(model, processor, spec, image_path, system_prompt, user_prompt,
                  max_new_tokens):
    prompt_text = render_prompt(processor, spec, system_prompt, user_prompt, image_path)
    target_device = model_device(model)
    inputs = prepare_inputs(processor, prompt_text, image_path, target_device)
    prompt_length = inputs["input_ids"].shape[-1]
    stopping = None
    if USE_JSON_STOP_CRITERIA:
        stopping = StoppingCriteriaList(
            [BalancedJsonStop(processor.tokenizer, prompt_length)])
    output_ids = model.generate(
        **inputs, do_sample=False, max_new_tokens=max_new_tokens,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        stopping_criteria=stopping, use_cache=True,
    )
    completion_ids = output_ids[0, prompt_length:]
    text = processor.decode(completion_ids, skip_special_tokens=True).strip()
    return text, int(completion_ids.shape[-1])


def extract_json_object(text):
    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    obj, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(obj, dict):
        raise ValueError("Prediction is not a JSON object")
    return obj


def safe_parse(text):
    try:
        return extract_json_object(text), None
    except Exception as error:
        return None, f"{type(error).__name__}: {error}"


def count_json_objects(text):
    # E6-Q measurement: how many complete top-level objects did the model emit?
    count, depth, in_string, escape = 0, 0, False, False
    for character in text:
        if escape:
            escape = False
            continue
        if character == "\\":
            escape = True
            continue
        if character == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                count += 1
    return count

## 5. Token-probability scoring (protocol 7.2)

### COVID: exact sequence likelihood over two candidates

For each candidate completion, teacher-force `prompt + candidate` and sum the log-probabilities
of the candidate's tokens. Then

    p(COVID) = exp(logL_yes) / (exp(logL_yes) + exp(logL_no))

computed in log space. Length normalisation is **not** applied: both candidates are the same
JSON with one word swapped, so a length correction would only add an arbitrary constant.

### mRALE: field-position distributions

Teacher-force the model's own generated JSON, locate the token positions of the four numerical
fields, and read the distribution over the digit tokens allowed at each position (extent 0-4,
density 0-3). That yields a per-component categorical distribution from a single forward pass,
which gives a soft expectation, a variance for the E3c uncertainty channel, and a continuous
severity score for rank-based external analysis (X3).

In [ ]:
@torch.inference_mode()
def sequence_log_likelihood(model, processor, spec, image_path, system_prompt, user_prompt,
                            candidate_text):
    """Sum of log p(token) over the candidate's tokens, teacher-forced after the prompt."""
    prompt_text = render_prompt(processor, spec, system_prompt, user_prompt, image_path)
    target_device = model_device(model)
    full = prepare_inputs(processor, prompt_text + candidate_text, image_path, target_device)
    prompt_only = prepare_inputs(processor, prompt_text, image_path, target_device)
    prompt_length = prompt_only["input_ids"].shape[-1]
    total_length = full["input_ids"].shape[-1]
    if total_length <= prompt_length:
        return float("nan"), 0

    logits = model(**full).logits.float()
    # logits[:, t] predicts token t+1, so the candidate's first token is predicted at
    # position prompt_length - 1.
    log_probs = torch.log_softmax(logits[0, prompt_length - 1:total_length - 1], dim=-1)
    targets = full["input_ids"][0, prompt_length:total_length]
    token_log_probs = log_probs.gather(-1, targets[:, None]).squeeze(-1)
    return float(token_log_probs.sum()), int(targets.shape[0])


def covid_probability_from_scoring(model, processor, spec, image_path, system_prompt,
                                   user_prompt):
    scores = {}
    for value in COVID_CANDIDATES:
        log_likelihood, n_tokens = sequence_log_likelihood(
            model, processor, spec, image_path, system_prompt, user_prompt,
            covid_candidate_json(value))
        scores[value] = {"log_likelihood": log_likelihood, "n_tokens": n_tokens}
    yes, no = scores["Yes"]["log_likelihood"], scores["No"]["log_likelihood"]
    if math.isnan(yes) or math.isnan(no):
        return None, scores
    maximum = max(yes, no)
    probability = math.exp(yes - maximum) / (math.exp(yes - maximum) + math.exp(no - maximum))
    return probability, scores


DIGIT_RANGES = {
    "extent_right_numerical": 5, "density_right_numerical": 4,
    "extent_left_numerical": 5, "density_left_numerical": 4,
}


@torch.inference_mode()
def score_numeric_fields(model, processor, spec, image_path, system_prompt, user_prompt,
                         generated_json_text):
    """
    Read the categorical distribution over the digit tokens at each numeric field position of
    the model's own generated JSON. One forward pass for all four fields.
    """
    prompt_text = render_prompt(processor, spec, system_prompt, user_prompt, image_path)
    target_device = model_device(model)
    tokenizer = processor.tokenizer

    full = prepare_inputs(
        processor, prompt_text + generated_json_text, image_path, target_device)
    prompt_only = prepare_inputs(processor, prompt_text, image_path, target_device)
    prompt_length = prompt_only["input_ids"].shape[-1]
    total_length = full["input_ids"].shape[-1]
    if total_length <= prompt_length:
        return {}

    logits = model(**full).logits.float()
    completion_ids = full["input_ids"][0, prompt_length:total_length]

    # Map each completion token to its character offset by incremental decoding. Slower than
    # offset mappings but works for every tokenizer, including those without fast variants.
    offsets, cursor = [], 0
    for index in range(completion_ids.shape[0]):
        piece = tokenizer.decode(completion_ids[index:index + 1], skip_special_tokens=True)
        offsets.append((cursor, cursor + len(piece)))
        cursor += len(piece)
    decoded = tokenizer.decode(completion_ids, skip_special_tokens=True)

    results = {}
    for field, n_classes in DIGIT_RANGES.items():
        match = re.search(rf'"{re.escape(field)}"\s*:\s*(-?\d+)', decoded)
        if not match:
            continue
        value_start = match.start(1)
        token_index = next(
            (index for index, (start, end) in enumerate(offsets)
             if start <= value_start < end), None)
        if token_index is None:
            continue
        position = prompt_length + token_index - 1
        if position < 0 or position >= logits.shape[1]:
            continue
        distribution = torch.log_softmax(logits[0, position], dim=-1)

        candidate_log_probs = []
        for value in range(n_classes):
            token_ids = tokenizer.encode(str(value), add_special_tokens=False)
            if len(token_ids) != 1:
                candidate_log_probs.append(float("-inf"))
                continue
            candidate_log_probs.append(float(distribution[token_ids[0]]))
        if all(math.isinf(value) for value in candidate_log_probs):
            continue
        tensor = torch.tensor(candidate_log_probs)
        probabilities = torch.softmax(tensor, dim=0).numpy()
        values = np.arange(n_classes, dtype=float)
        mean = float((probabilities * values).sum())
        results[field] = {
            "distribution": [round(float(p), 6) for p in probabilities],
            "expected": mean,
            "variance": float((probabilities * (values - mean) ** 2).sum()),
            "argmax": int(np.argmax(probabilities)),
        }
    return results


def mrale_from_parsed(parsed):
    """Extract integer components + total from a parsed JSON object, with validation."""
    if not isinstance(parsed, dict):
        return None
    values = {}
    for field, n_classes in DIGIT_RANGES.items():
        raw = parsed.get(field)
        try:
            value = int(raw)
        except (TypeError, ValueError):
            return None
        if not (0 <= value < n_classes):
            return None
        values[field] = value
    right = values["extent_right_numerical"] * values["density_right_numerical"]
    left = values["extent_left_numerical"] * values["density_left_numerical"]
    reported = parsed.get("mRALE Score")
    try:
        reported_total = int(reported)
    except (TypeError, ValueError):
        reported_total = None
    return {
        "extent_right": values["extent_right_numerical"],
        "density_right": values["density_right_numerical"],
        "extent_left": values["extent_left_numerical"],
        "density_left": values["density_left_numerical"],
        "mrale_right": right, "mrale_left": left,
        # Protocol Section 3.2: the total is CONSTRAINED to right + left. The model's own
        # reported total is kept separately so formula consistency stays measurable.
        "mrale_total": right + left,
        "reported_total": reported_total,
        "formula_consistent": reported_total == right + left,
    }

## 6. Build the work list

In [ ]:
def load_view_index():
    path = NB04_DIR / "view_index.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} not found. Run Stage A NB 04 first.")
    frame = pd.read_csv(path)
    print(f"view_index.csv: {len(frame):,} rows, cohorts={dict(Counter(frame['cohort']))}")
    return frame


def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level."
        )
    frame = pd.read_csv(path)
    print(f"midrc_folds_v2.csv: {len(frame):,} images, "
          f"{frame['group_id'].nunique():,} groups, folds={dict(sorted(Counter(frame['fold']).items()))}")
    return frame


def build_cohort_table():
    # One row per image: labels + fold + every cached view path. This is the single table
    # every Stage B notebook trains and predicts from.
    views = load_view_index()
    folds = load_folds()

    internal = folds.merge(
        views[views["cohort"] == "MIDRC"].drop(columns=["held_out_fold"], errors="ignore"),
        on="filename", how="inner", suffixes=("", "_view"),
    )
    if len(internal) != len(folds):
        missing = set(folds["filename"]) - set(internal["filename"])
        raise RuntimeError(
            f"{len(missing)} fold images have no NB 04 localization row (e.g. "
            f"{sorted(missing)[:5]}). Re-run NB 04 with MAX_IMAGES=None."
        )
    internal["mrale_right"] = (internal["extent_right_numerical"]
                               * internal["density_right_numerical"])
    internal["mrale_left"] = (internal["extent_left_numerical"]
                              * internal["density_left_numerical"])
    internal["is_external"] = False

    external_rows = []
    external_dir = NB03_DIR / "external_manifests"
    if external_dir.is_dir():
        for manifest_path in sorted(external_dir.glob("*_manifest.csv")):
            frame = pd.read_csv(manifest_path)
            if "status" in frame.columns:
                frame = frame[frame["status"] == "OK"]
            if not len(frame):
                continue
            cohort = str(frame["cohort"].iloc[0])
            merged = frame.merge(
                views[views["cohort"] == cohort][
                    ["filename", "v0_image", "v1_thorax_image", "v2_left_image",
                     "v2_right_image", "left_box", "right_box", "any_fallback"]
                ],
                on="filename", how="inner",
            )
            merged["fold"] = -1
            merged["is_external"] = True
            merged["group_id"] = "external::" + merged["filename"].astype(str)
            for column in ["mrale_total_annotated", "mrale_right", "mrale_left",
                           "extent_right_numerical", "density_right_numerical",
                           "extent_left_numerical", "density_left_numerical"]:
                if column not in merged.columns:
                    merged[column] = np.nan
            if "mrale_total" in merged.columns:
                merged["mrale_total_annotated"] = merged["mrale_total"]
            external_rows.append(merged)

    table = pd.concat([internal] + external_rows, ignore_index=True, sort=False)
    table["image_key"] = table.apply(
        lambda row: f"{row.get('cohort', 'MIDRC')}::{row['filename']}", axis=1)
    print()
    print(f"Cohort table: {len(table):,} rows "
          f"({int((~table['is_external']).sum()):,} internal, "
          f"{int(table['is_external'].sum()):,} external)")
    return table


def grouped_inner_split(subset, fraction, seed):
    # Group-aware inner validation split, same construction as the tested notebooks: whole
    # groups move together so the inner split cannot leak either.
    groups = sorted(subset["group_id"].astype(str).unique())
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_validation = max(1, round(len(groups) * fraction))
    validation_groups = set(groups[:n_validation])
    is_validation = subset["group_id"].astype(str).isin(validation_groups)
    train, validation = subset[~is_validation], subset[is_validation]
    assert not (set(train["group_id"]) & set(validation["group_id"]))
    return train, validation


def ground_truth_fields(row):
    def maybe_int(value):
        return None if value is None or (isinstance(value, float) and math.isnan(value)) else int(value)
    covid = row.get("covid_positive")
    if isinstance(covid, float) and math.isnan(covid):
        covid = None
    return {
        "gt_covid": covid if covid in {"Yes", "No"} else None,
        "gt_mrale_total": maybe_int(row.get("mrale_total_annotated")),
        "gt_mrale_right": maybe_int(row.get("mrale_right")),
        "gt_mrale_left": maybe_int(row.get("mrale_left")),
        "gt_extent_right": maybe_int(row.get("extent_right_numerical")),
        "gt_density_right": maybe_int(row.get("density_right_numerical")),
        "gt_extent_left": maybe_int(row.get("extent_left_numerical")),
        "gt_density_left": maybe_int(row.get("density_left_numerical")),
    }


def evaluate_arm(rows, label):
    # Single entry point for metrics, so every arm in Table 2 is scored identically.
    covid_rows = [row for row in rows if row.get("gt_covid") is not None]
    metrics = {"arm": label, "n_rows": len(rows)}
    if covid_rows:
        metrics["covid"] = cm.classification_metrics(
            [row["gt_covid"] for row in covid_rows],
            [row.get("covid_pred") for row in covid_rows],
            [row.get("covid_score") for row in covid_rows],
        )
    mrale_rows = [row for row in rows if row.get("gt_mrale_total") is not None]
    if mrale_rows:
        metrics["mrale"] = cm.mrale_metrics(mrale_rows)
    metrics["output"] = cm.localization_free_metrics(rows)
    return metrics


def print_arm_summary(metrics):
    covid = metrics.get("covid", {})
    mrale = metrics.get("mrale", {})
    print(f"  {metrics['arm']:<34} "
          f"AUROC={covid.get('auroc', float('nan')):.4f} "
          f"balAcc={covid.get('balanced_accuracy', float('nan')):.4f} "
          f"spec={covid.get('specificity', float('nan')):.4f} | "
          f"mRALE MAE={mrale.get('mae', float('nan')):.3f} "
          f"QWK={mrale.get('qwk', float('nan')):.4f} "
          f"cov={mrale.get('coverage', float('nan')):.3f}")

In [ ]:
cohort = build_cohort_table()
if not INCLUDE_EXTERNAL:
    cohort = cohort[~cohort["is_external"]]
work = cohort.reset_index(drop=True)
if MAX_IMAGES_PER_ARM is not None:
    internal_sample = work[~work["is_external"]].groupby("fold", group_keys=False).head(
        max(2, MAX_IMAGES_PER_ARM // N_FOLDS))
    work = pd.concat([internal_sample, work[work["is_external"]].head(4)], ignore_index=True)
    print(f"WIRING CHECK: {len(work)} images")

work = work.sort_values("image_key").reset_index(drop=True)
print(f"Work list: {len(work):,} images, "
      f"{len(RUN_ARMS)} arms, {len(TASKS)} tasks "
      f"-> {len(work) * len(RUN_ARMS) * len(TASKS):,} generations")
if RUN_TOKEN_SCORING:
    print(f"Plus scoring passes: ~{len(work) * len(RUN_ARMS) * (len(COVID_CANDIDATES) + 1):,}")

## 6b. Image-binding preflight — run this before any long generation

A vision-language model that never receives its image does not error. It produces fluent,
confident text — `"Please provide a frontal chest X-ray image."` — and every JSON parse fails.
From the metrics alone that is indistinguishable from a weak model, which is how a 20-hour run
can complete with `valid_rate = 0.000` and look like a result.

The cause is subtle. `{"type": "image"}` with no value is enough for MedGemma and Qwen3.5, whose
chat templates insert a vision token for it. Some templates — notably Qwen2.5-VL derivatives such
as NV-Reason-CXR-3B — emit **nothing** for a valueless image entry. The rendered prompt then
contains no vision placeholder, the image is never attended to, and the model asks for it.

This cell checks, per arm, in a few seconds each:

1. the rendered prompt contains one of the model's image placeholder tokens;
2. the processor output actually carries vision tensors (`pixel_values` / `image_grid_thw`);
3. one real CXR generates something that is **not** an image-absent refusal.

`render_prompt` now tries a valueless entry, then a valued one, then a text-only fallback, and
keeps the first that produces a placeholder — so for most models this repairs itself and the cell
simply confirms it. Which variant was used is recorded per arm in `image_binding_preflight.csv`
and in `run_config.json`.


In [ ]:
KNOWN_INCOMPATIBLE_OUTPUT_FORMAT = {
    "E0c_nvreason": "emits <think>...</think><answer>labels</answer>, never JSON; "
                    "image binding is confirmed working (see nvreason_finding.json)",
}

preflight_rows = []

if RUN_IMAGE_PREFLIGHT:
    probe_row = work[~work["is_external"]].iloc[0]
    probe_image = probe_row["v0_image"]
    print(f"Preflight image: {probe_row['image_key']}")
    print()

    for arm in RUN_ARMS:
        spec = MODEL_ARMS[arm]
        entry = {"arm": arm, "model_id": spec["model_id"]}
        model = None
        try:
            model, processor_arm, revision = load_vlm(spec)
            system_prompt = templates["covid_classification"]["system"]
            user_prompt = (templates["covid_classification"]["user"]
                           + ARM_PROMPT_SUFFIX.get(arm, ""))

            prompt_text = render_prompt(
                processor_arm, spec, system_prompt, user_prompt, probe_image)
            entry["template_variant"] = IMAGE_TEMPLATE_VARIANT.get(spec["model_id"])
            entry["placeholder_in_prompt"] = has_image_placeholder(prompt_text)
            entry["placeholders_found"] = ";".join(
                token for token in IMAGE_PLACEHOLDER_TOKENS if token in prompt_text)

            inputs = prepare_inputs(processor_arm, prompt_text, probe_image,
                                    model_device(model))
            vision_keys = [key for key in inputs
                           if key in {"pixel_values", "image_grid_thw", "pixel_attention_mask",
                                      "image_sizes", "aspect_ratio_ids"}]
            entry["vision_tensors"] = ";".join(vision_keys)
            entry["has_vision_tensors"] = bool(vision_keys)
            entry["prompt_tokens"] = int(inputs["input_ids"].shape[-1])

            text, n_tokens = generate_text(
                model, processor_arm, spec, probe_image, system_prompt, user_prompt,
                MAX_NEW_TOKENS["covid_classification"])
            entry["completion"] = text[:200]
            entry["completion_tokens"] = n_tokens
            lowered = text.lower()
            entry["looks_image_absent"] = any(
                phrase in lowered for phrase in IMAGE_ABSENT_PHRASES)
            entry["parses_as_json"] = False
            try:
                extract_json_object(text)
                entry["parses_as_json"] = True
            except Exception:
                pass
            entry["status"] = (
                "OK" if (entry["placeholder_in_prompt"] and entry["has_vision_tensors"]
                         and not entry["looks_image_absent"])
                else "IMAGE NOT BOUND")
        except Exception as exc:
            entry["status"] = "LOAD/RUN FAILED"
            entry["error"] = f"{type(exc).__name__}: {exc}"
        finally:
            if model is not None:
                del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        preflight_rows.append(entry)
        print(f"--- {arm}  [{entry['status']}]")
        print(f"    template variant   : {entry.get('template_variant')}")
        print(f"    placeholder present: {entry.get('placeholder_in_prompt')} "
              f"({entry.get('placeholders_found') or 'none'})")
        print(f"    vision tensors     : {entry.get('vision_tensors') or 'NONE'}")
        print(f"    prompt tokens      : {entry.get('prompt_tokens')}")
        print(f"    completion         : {entry.get('completion')!r}")
        print(f"    parses as JSON     : {entry.get('parses_as_json')}")
        if entry.get("error"):
            print(f"    error              : {entry['error']}")
        print()

    preflight = pd.DataFrame(preflight_rows)
    preflight.to_csv(NB07_DIR / "image_binding_preflight.csv", index=False)

    # ---- Cache-consistency guard ----------------------------------------------------------
    # render_prompt now chooses among several message schemas. If it resolves a DIFFERENT
    # schema than the one a cached arm was generated with, the cached predictions came from a
    # different prompt and must not be mixed with new ones. Recorded and compared explicitly,
    # because a silent prompt change is the kind of thing that is invisible in the metrics.
    recorded = _load_status().get("image_template_variant", {})
    resolved = {row["arm"]: row.get("template_variant") for row in preflight_rows}
    drifted = [arm for arm, variant in resolved.items()
               if arm in recorded and recorded[arm] != variant]
    if drifted:
        print("=" * 78)
        print(f"PROMPT RENDERING CHANGED for {drifted}:")
        for arm in drifted:
            print(f"    {arm}: cache built with {recorded[arm]!r}, now resolves "
                  f"{resolved[arm]!r}")
        print("  Cached predictions for these arms came from a different prompt. Either pin the")
        print("  original schema via ARM_MESSAGE_SCHEMA, or regenerate with")
        print(f"      RESET_ARMS = {drifted}")
        print("  Do not mix them: half a table computed under each prompt is not a result.")
        raise RuntimeError(
            f"Template variant drifted for {drifted}: "
            + "; ".join(f"{arm} {recorded[arm]!r} -> {resolved[arm]!r}" for arm in drifted)
            + ". Pin ARM_MESSAGE_SCHEMA or use RESET_ARMS.")

    status = _load_status()
    status["image_template_variant"] = {**recorded, **resolved}
    _save_status(status)

# An arm whose image binds but whose OUTPUT FORMAT differs is not a binding failure. Listing
# it here records the distinction: NV-Reason emits <think>/<answer>, never JSON, so it will
# always "refuse" a JSON schema while reading the radiograph perfectly well. See section 9b.
    broken = [row["arm"] for row in preflight_rows
              if row["status"] != "OK"
              and not (row.get("placeholder_in_prompt") and row.get("has_vision_tensors")
                       and row["arm"] in KNOWN_INCOMPATIBLE_OUTPUT_FORMAT)]
    tolerated = [row["arm"] for row in preflight_rows
                 if row["status"] != "OK" and row["arm"] not in broken]
    for arm in tolerated:
        print(f"  NOTE {arm}: image binding OK, but its output format is known to differ "
              f"({KNOWN_INCOMPATIBLE_OUTPUT_FORMAT.get(arm)}).")
        print("       Not a binding failure; see section 9b for the analysis and the honest")
        print("       reporting position.")
    if broken:
        print("=" * 78)
        print(f"IMAGE BINDING BROKEN FOR: {broken}")
        print()
        print("Do NOT start generation for these arms. Whatever they produce will be text")
        print("written without looking at the radiograph, and it will not be distinguishable")
        print("from a weak result once it is in the metrics table.")
        print()
        print("If placeholder_in_prompt is False, the chat template emits nothing for the")
        print("image entry. render_prompt already tries a valued entry and a text-only")
        print("fallback; if none produced a placeholder, that model needs its own message")
        print("format -- check its model card for the expected content schema.")
        print()
        print("If placeholder_in_prompt is True but has_vision_tensors is False, the processor")
        print("is not consuming the PIL image; check whether it expects a list rather than a")
        print("single image, or a different keyword than images=.")
        print()
        print("After fixing, use RESET_ARMS to discard that arm's cached predictions:")
        print(f"    RESET_ARMS = {broken}")
        detail_lines = []
        for row in preflight_rows:
            if row["status"] == "OK":
                continue
            detail_lines.append(
                f"  {row['arm']}: status={row['status']} "
                f"placeholder={row.get('placeholder_in_prompt')} "
                f"vision_tensors={row.get('vision_tensors') or 'NONE'} "
                f"template_variant={row.get('template_variant')} "
                f"refusal={row.get('looks_image_absent')} "
                f"completion={row.get('completion')!r}"
                + (f" error={row.get('error')}" if row.get("error") else ""))
        # Reasons go in the message so a pasted traceback is self-explanatory.
        raise RuntimeError(
            "Image binding preflight failed for "
            f"{broken}.\n" + "\n".join(detail_lines) + "\n"
            "Run the schema probe in section 6c: it tries every known message convention for "
            "these arms and pins the first that works. See image_binding_preflight.csv."
        )
    print("All arms bind the image correctly. Safe to generate.")
else:
    print("RUN_IMAGE_PREFLIGHT is False -- skipping. Not recommended: an unbound image")
    print("produces fluent text with zero valid JSON and looks like a weak model, not a bug.")


### 6c. Schema probe — for an arm the preflight still rejects

If 6b rejects an arm, this cell loads that model **once** and tries every known message
convention end to end: render the prompt, check for a vision placeholder, check the processor
emits vision tensors, generate on one real CXR, and check the reply is not an image-absent
refusal. It prints a table and pins the first fully-working schema into `ARM_MESSAGE_SCHEMA`.

The schemas tried, in order of increasing intervention:

| schema | form |
| --- | --- |
| `bare` | `{"type": "image"}` with no value |
| `valued` | `{"type": "image", "image": "/path/img.png"}` |
| `file_uri` | `{"type": "image", "image": "file:///path/img.png"}` — the documented Qwen2.5-VL form |
| `system_as_string` | plain-string system turn instead of a content list |
| `system_folded_into_user` | no system turn at all; instructions moved into the user turn |
| `manual_vision_tokens` | the model's own vision token (e.g. `<\|vision_start\|><\|image_pad\|><\|vision_end\|>`) injected directly into the user text |

`manual_vision_tokens` is the guaranteed fallback. If no message form makes the template emit a
placeholder, we insert the token ourselves rather than let the model run blind — the token is
discovered from the processor and tokenizer vocabulary, not hardcoded.

Two things worth noting. A schema that yields a placeholder but no vision tensors means the
*processor* is not consuming the image, which is a different fault from the template. And a
schema that passes both but still returns a refusal means the image reaches the model but is not
being attended to — check the model card for a required image resolution or a
`min_pixels`/`max_pixels` setting.


In [ ]:
PROBE_ARMS = [row["arm"] for row in preflight_rows
              if row.get("status") != "OK"] if "preflight_rows" in dir() else []

schema_probe_rows = []
if PROBE_ARMS:
    probe_row = work[~work["is_external"]].iloc[0]
    probe_image = probe_row["v0_image"]
    print(f"Probing message schemas for {PROBE_ARMS} on {probe_row['image_key']}")
    print()

    for arm in PROBE_ARMS:
        spec = dict(MODEL_ARMS[arm])
        spec["_arm"] = arm
        model = None
        try:
            model, processor_arm, _ = load_vlm(spec)
            token = vision_token_string(processor_arm, model)
            print(f"=== {arm}  ({spec['model_id']})")
            print(f"    discovered vision token: {token!r}")
            system_prompt = templates["covid_classification"]["system"]
            user_prompt = (templates["covid_classification"]["user"]
                           + ARM_PROMPT_SUFFIX.get(arm, ""))

            for schema in MESSAGE_SCHEMAS:
                entry = {"arm": arm, "schema": schema}
                try:
                    messages, note = build_messages(
                        schema, system_prompt, user_prompt, probe_image, processor_arm, model)
                    entry["note"] = note
                    text = _apply_template(processor_arm, spec, messages)
                    entry["placeholder"] = has_image_placeholder(text)
                    entry["prompt_head"] = text[:90].replace("\n", "\\n")

                    inputs = prepare_inputs(processor_arm, text, probe_image,
                                            model_device(model))
                    vision_keys = [key for key in inputs
                                   if key in {"pixel_values", "image_grid_thw",
                                              "pixel_attention_mask", "image_sizes"}]
                    entry["vision_tensors"] = ";".join(vision_keys) or None
                    entry["prompt_tokens"] = int(inputs["input_ids"].shape[-1])

                    if entry["placeholder"] and vision_keys:
                        completion, _ = generate_text(
                            model, processor_arm, spec, probe_image, system_prompt,
                            user_prompt, MAX_NEW_TOKENS["covid_classification"])
                        entry["completion"] = completion[:120]
                        lowered = completion.lower()
                        entry["refusal"] = any(phrase in lowered
                                               for phrase in IMAGE_ABSENT_PHRASES)
                        try:
                            extract_json_object(completion)
                            entry["parses"] = True
                        except Exception:
                            entry["parses"] = False
                        entry["works"] = bool(not entry["refusal"])
                    else:
                        entry["works"] = False
                except Exception as exc:
                    entry["error"] = f"{type(exc).__name__}: {exc}"
                    entry["works"] = False
                schema_probe_rows.append(entry)
                flag = "WORKS" if entry.get("works") else "no"
                print(f"    {schema:<26} placeholder={entry.get('placeholder')} "
                      f"vision={entry.get('vision_tensors')} -> {flag}")
                if entry.get("completion"):
                    print(f"      completion: {entry['completion']!r}")
                if entry.get("error"):
                    print(f"      error: {entry['error']}")

            winners = [row for row in schema_probe_rows
                       if row["arm"] == arm and row.get("works")]
            if winners:
                # Prefer the least invasive schema that fully works.
                chosen = min(winners, key=lambda row: MESSAGE_SCHEMAS.index(row["schema"]))
                ARM_MESSAGE_SCHEMA[spec["model_id"]] = chosen["schema"]
                ARM_MESSAGE_SCHEMA[arm] = chosen["schema"]
                print()
                print(f"    ADOPTED: {chosen['schema']}  ({chosen.get('note')})")
                print("    Pin it by adding to the config cell:")
                print(f"        ARM_MESSAGE_SCHEMA = {{\"{spec['model_id']}\": "
                      f"\"{chosen['schema']}\"}}")
                print(f"    Then: RESET_ARMS = ['{arm}']  and re-run 6b -> 7.")
            else:
                print()
                print(f"    NO SCHEMA WORKS for {arm}.")
                print("    Every convention either failed to emit a placeholder, failed to")
                print("    produce vision tensors, or still returned an image-absent refusal.")
                print("    Next steps, in order:")
                print("      1. Read the model card for its required message format and any")
                print("         min_pixels/max_pixels or image-resolution constraints.")
                print("      2. Check whether it needs a helper such as qwen_vl_utils'")
                print("         process_vision_info to extract images before processor(...).")
                print("      3. If it cannot be made to bind reliably, DROP the arm and say so:")
                print("         a model that cannot see the image is not a baseline, and an")
                print("         honest omission is better than a fabricated row in Table 2.")
        finally:
            if model is not None:
                del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        print()

    probe = pd.DataFrame(schema_probe_rows)
    probe.to_csv(NB07_DIR / "message_schema_probe.csv", index=False)
    print("Wrote message_schema_probe.csv")
    print()
    print("Adopted schemas:", {k: v for k, v in ARM_MESSAGE_SCHEMA.items() if "/" in k})
else:
    print("No arm needs probing: 6b passed for all arms (or has not been run yet).")


## 7. Run each arm

Checkpointed per (arm, image, task): interrupting and re-running resumes. Models are loaded one
at a time and released, because three 3-4B models will not co-reside comfortably with the
scoring passes' activation memory.

In [ ]:
PREDICTION_PATH = NB07_DIR / "predictions_zeroshot.jsonl"
TRACE_PATH = NB07_DIR / "nvreason_traces.jsonl"
SCORE_RECORDS_PATH = NB07_DIR / "score_records.jsonl"

completed = cm.load_jsonl_by_key(PREDICTION_PATH, ["arm", "image_key", "task"])
print(f"Resuming with {len(completed):,} completed (arm, image, task) rows.")

GENERATION_FINGERPRINT = fingerprint(
    sorted(RUN_ARMS), sorted(TASKS), len(work), RUN_TOKEN_SCORING, SCORE_MRALE_FIELDS,
    USE_JSON_STOP_CRITERIA, MAX_NEW_TOKENS,
    {arm: MODEL_REVISIONS.get(MODEL_ARMS[arm]["model_id"]) for arm in RUN_ARMS},
    ARM_PROMPT_SUFFIX,
    # Message schema affects the prompt, so it belongs in the fingerprint.
    {arm: ARM_MESSAGE_SCHEMA.get(MODEL_ARMS[arm]["model_id"]) for arm in RUN_ARMS},
)
expected_rows = len(work) * len(RUN_ARMS) * len(TASKS)

# Score records are persisted so section 8 never needs the models reloaded.
score_records = cm.read_jsonl(SCORE_RECORDS_PATH) if SCORE_RECORDS_PATH.is_file() else []


def rebuild_score_records(completed_rows):
    """
    Recover score records from the saved prediction rows.

    Earlier versions of this notebook kept score records in memory only, so a run completed
    under that version loses them on kernel restart -- which then looks like "token scoring
    produced no validated scores" even though the work was done. The scores were always
    persisted inside the prediction rows (covid_score and
    extra.candidate_log_likelihoods), so they can be reconstructed exactly, with no recompute.
    """
    recovered = []
    for key, row in completed_rows.items():
        arm, image_key, task = key
        if task != "covid_classification":
            continue
        probability = row.get("covid_score")
        if probability is None:
            continue
        candidates = (row.get("extra") or {}).get("candidate_log_likelihoods") or {}
        yes = (candidates.get("Yes") or {}).get("log_likelihood")
        no = (candidates.get("No") or {}).get("log_likelihood")
        recovered.append({
            "arm": arm, "image_key": image_key, "covid_score": probability,
            "logL_yes": yes, "logL_no": no,
            "generated_decision": row.get("covid_pred"),
            "scored_decision": "Yes" if probability >= 0.5 else "No",
            "gt_covid": row.get("gt_covid"),
            "recovered_from_predictions": True,
        })
    return recovered


scored_rows_present = sum(
    1 for key, row in completed.items()
    if key[2] == "covid_classification" and row.get("covid_score") is not None)
if len(score_records) < scored_rows_present:
    have = {(row["arm"], row["image_key"]) for row in score_records}
    recovered = [row for row in rebuild_score_records(completed)
                 if (row["arm"], row["image_key"]) not in have]
    if recovered:
        for row in recovered:
            cm.append_jsonl(SCORE_RECORDS_PATH, row)
        score_records.extend(recovered)
        print(f"BACKFILL: recovered {len(recovered):,} score records from "
              "predictions_zeroshot.jsonl and persisted them to score_records.jsonl.")
        print("  (A run completed under an earlier version of this notebook kept these in "
              "memory only; nothing was recomputed.)")

if RESET_ARMS:
    dropped = [key for key in completed if key[0] in set(RESET_ARMS)]
    for key in dropped:
        del completed[key]
    print(f"RESET_ARMS={RESET_ARMS}: discarded {len(dropped):,} cached rows; those arms will "
          "regenerate. Re-appended rows supersede the old ones in the JSONL.")

if (not FORCE_REGENERATE
        and stage_complete("generation", GENERATION_FINGERPRINT, [PREDICTION_PATH])
        and len(completed) >= expected_rows):
    print()
    print(f"SKIPPING SECTION 7: generation already complete for this configuration "
          f"({len(completed):,} rows, {len(score_records):,} score records).")
    print("Set FORCE_REGENERATE = True to redo it. This is the ~20-hour section.")
    SKIP_GENERATION = True
else:
    SKIP_GENERATION = False
    if len(completed):
        print(f"Partial run detected: {len(completed):,} of {expected_rows:,} rows present; "
              "continuing from where it stopped.")

for arm in ([] if SKIP_GENERATION else RUN_ARMS):
    spec = MODEL_ARMS[arm]
    pending = [
        row for _, row in work.iterrows()
        if any((arm, str(row["image_key"]), task) not in completed for task in TASKS)
    ]
    if not pending:
        print(f"{arm}: already complete, skipping model load.")
        continue

    print("=" * 78)
    print(f"ARM {arm}: {spec['model_id']}  ({len(pending):,} images pending)")
    model, processor, revision = load_vlm(spec)
    n_parameters = sum(p.numel() for p in model.parameters())
    print(f"  loaded: {n_parameters / 1e9:.2f}B parameters, dtype={DTYPE}")

    started = time.perf_counter()
    for position, row in enumerate(pending, start=1):
        image_key = str(row["image_key"])
        image_path = row["v0_image"]
        truth = ground_truth_fields(row)

        for task in TASKS:
            key = (arm, image_key, task)
            if key in completed:
                continue
            system_prompt = templates[task]["system"]
            user_prompt = templates[task]["user"] + ARM_PROMPT_SUFFIX.get(arm, "")
            image_started = time.perf_counter()

            try:
                text, n_completion_tokens = generate_text(
                    model, processor, spec, image_path, system_prompt, user_prompt,
                    MAX_NEW_TOKENS[task])
            except Exception as exc:
                text, n_completion_tokens = "", 0
                parse_error = f"generation_failed: {type(exc).__name__}: {exc}"
                parsed = None
            else:
                parsed, parse_error = safe_parse(text)

            record = {
                "image_key": image_key, "cohort": row.get("cohort", "MIDRC"),
                "subcohort": row.get("subcohort", "MIDRC"), "filename": row["filename"],
                "held_out_fold": (None if row.get("is_external")
                                  else int(row["fold"]) if not pd.isna(row.get("fold")) else None),
                "agent": spec["agent"], "arm": arm, "view": "v0", "task": task,
                "model_id": spec["model_id"], "model_revision": revision,
                "raw_output": text[:4000], "parse_error": parse_error,
                "seconds": round(time.perf_counter() - image_started, 4),
                **truth,
            }
            extra = {
                "n_completion_tokens": n_completion_tokens,
                "n_json_objects": count_json_objects(text),
            }

            if task == "covid_classification":
                value = None
                if isinstance(parsed, dict):
                    raw = str(parsed.get("covid_positive", "")).strip().lower()
                    value = "Yes" if raw in {"yes", "true", "1", "positive"} else (
                        "No" if raw in {"no", "false", "0", "negative"} else None)
                record["covid_pred"] = value
                record["valid"] = value is not None
                if value is None and parse_error is None:
                    record["parse_error"] = "covid_value_not_recognised"

                if RUN_TOKEN_SCORING:
                    try:
                        probability, candidate_scores = covid_probability_from_scoring(
                            model, processor, spec, image_path, system_prompt, user_prompt)
                        record["covid_score"] = probability
                        extra["candidate_log_likelihoods"] = candidate_scores
                        if probability is not None:
                            score_records.append({
                                "arm": arm, "image_key": image_key,
                                "covid_score": probability,
                                "logL_yes": candidate_scores["Yes"]["log_likelihood"],
                                "logL_no": candidate_scores["No"]["log_likelihood"],
                                "generated_decision": value,
                                "scored_decision": "Yes" if probability >= 0.5 else "No",
                                "gt_covid": truth["gt_covid"],
                            })
                    except Exception as exc:
                        extra["scoring_error"] = f"{type(exc).__name__}: {exc}"

            else:
                decoded = mrale_from_parsed(parsed)
                if decoded is None:
                    record["valid"] = False
                    if parse_error is None:
                        record["parse_error"] = "mrale_fields_missing_or_out_of_range"
                else:
                    record.update({
                        "mrale_total": decoded["mrale_total"],
                        "mrale_right": decoded["mrale_right"],
                        "mrale_left": decoded["mrale_left"],
                        "extent_right": decoded["extent_right"],
                        "density_right": decoded["density_right"],
                        "extent_left": decoded["extent_left"],
                        "density_left": decoded["density_left"],
                        "valid": True,
                    })
                    extra["reported_total"] = decoded["reported_total"]
                    extra["formula_consistent"] = decoded["formula_consistent"]

                    if RUN_TOKEN_SCORING and SCORE_MRALE_FIELDS and text:
                        try:
                            fields = score_numeric_fields(
                                model, processor, spec, image_path, system_prompt,
                                user_prompt, text)
                            if fields:
                                extra["field_distributions"] = fields
                                expected_right = (
                                    fields.get("extent_right_numerical", {}).get("expected")
                                    or 0) * (
                                    fields.get("density_right_numerical", {}).get("expected") or 0)
                                expected_left = (
                                    fields.get("extent_left_numerical", {}).get("expected")
                                    or 0) * (
                                    fields.get("density_left_numerical", {}).get("expected") or 0)
                                record["mrale_total_expected"] = expected_right + expected_left
                                record["mrale_uncertainty"] = float(np.mean([
                                    item["variance"] for item in fields.values()]))
                        except Exception as exc:
                            extra["field_scoring_error"] = f"{type(exc).__name__}: {exc}"

            if spec.get("capture_traces") and task == "mrale_prediction":
                try:
                    trace_text, _ = generate_text(
                        model, processor, spec, image_path, system_prompt,
                        user_prompt + " Explain your reasoning before the JSON.",
                        TRACE_MAX_NEW_TOKENS)
                    cm.append_jsonl(TRACE_PATH, {
                        "arm": arm, "image_key": image_key, "trace": trace_text[:8000],
                        "gt_mrale_total": truth["gt_mrale_total"],
                        "gt_covid": truth["gt_covid"],
                    })
                except Exception as exc:
                    extra["trace_error"] = f"{type(exc).__name__}: {exc}"

            record["extra"] = extra
            prediction_row = cm.make_prediction_row(**record)
            cm.append_jsonl(PREDICTION_PATH, prediction_row)
            completed[key] = prediction_row
            if task == "covid_classification" and record.get("covid_score") is not None:
                # Persisted immediately so section 8 can run without reloading any model.
                cm.append_jsonl(SCORE_RECORDS_PATH, score_records[-1])

        if position % 25 == 0 or position == len(pending):
            elapsed = time.perf_counter() - started
            rate = position / max(elapsed, 1e-6)
            remaining = (len(pending) - position) / max(rate, 1e-9)
            print(f"    [{position}/{len(pending)}] {rate:.2f} img/s, "
                  f"~{remaining / 60:.1f} min remaining")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    print(f"  {arm} complete in {(time.perf_counter() - started) / 60:.1f} min")

if not SKIP_GENERATION:
    mark_stage("generation", GENERATION_FINGERPRINT,
               {"rows": len(completed), "score_records": len(score_records),
                "arms": RUN_ARMS})

predictions = list(completed.values())
print()
print(f"Total prediction rows: {len(predictions):,}")
print(f"Score records: {len(score_records):,}")

## 8. Validate the continuous score against the generated decision

Protocol 7.2's precondition. If `argmax` over the scored candidates disagrees with the model's
own free-generation decision on more than 0.5% of cases, the score is not a faithful
description of the model's behaviour and must not be used for ROC, PR, DeLong, or calibration.
This is checked before any AUROC is reported, not after.

In [ ]:
score_frame = pd.DataFrame(score_records)
validation_rows = []

if len(score_frame):
    try:
        score_frame.to_parquet(NB07_DIR / "token_probability_scores.parquet", index=False)
    except Exception as exc:
        print(f"Parquet unavailable ({exc}); writing CSV instead.")
        score_frame.to_csv(NB07_DIR / "token_probability_scores.csv", index=False)

    for arm in sorted(score_frame["arm"].unique()):
        subset = score_frame[
            (score_frame["arm"] == arm) & score_frame["generated_decision"].notna()]
        if not len(subset):
            continue
        agrees = subset["scored_decision"] == subset["generated_decision"]
        agreement = float(agrees.mean())
        margin = (subset["logL_yes"] - subset["logL_no"]).abs()
        decisive = margin >= AGREEMENT_DECISIVE_MARGIN
        decisive_agreement = float(agrees[decisive].mean()) if decisive.any() else float("nan")
        gating = (decisive_agreement
                  if GATE_ON_DECISIVE_AGREEMENT and not math.isnan(decisive_agreement)
                  else agreement)
        validation_rows.append({
            "arm": arm, "n_scored": len(subset),
            "score_generation_agreement": round(agreement, 5),
            "decisive_agreement": (None if math.isnan(decisive_agreement)
                                   else round(decisive_agreement, 5)),
            "n_decisive": int(decisive.sum()),
            "near_tie_fraction": round(float((~decisive).mean()), 4),
            "decisive_margin_nats": AGREEMENT_DECISIVE_MARGIN,
            "mean_abs_logL_margin": round(float(margin.mean()), 3),
            "mean_score": round(float(subset["covid_score"].mean()), 4),
            "score_std": round(float(subset["covid_score"].std()), 4),
            "gating_agreement": (None if math.isnan(gating) else round(gating, 5)),
            "meets_protocol_7_2": bool(
                (not math.isnan(gating)) and gating >= SCORE_AGREEMENT_THRESHOLD),
        })

    validation = pd.DataFrame(validation_rows)
    validation.to_csv(NB07_DIR / "score_validation.csv", index=False)
    print(validation.to_string(index=False))
    print()
    for row in validation_rows:
        if not row["meets_protocol_7_2"]:
            print(f"  {row['arm']}: agreement {row['score_generation_agreement']:.4f} is below "
                  "0.995. The scored probability does not reproduce the model's own decision, "
                  "so it must not be used for ROC/DeLong until the cause is found. Common "
                  "causes: the chat template inserting tokens between prompt and answer, or a "
                  "candidate string that is not token-identical to what the model emits.")
    if all(row["meets_protocol_7_2"] for row in validation_rows):
        print("  All arms meet protocol 7.2. AUROC, AUPRC, DeLong, and calibration are now")
        print("  available for generative models -- which is what referee 1.3 and 2d asked for")
        print("  and what the rejected submission could not supply.")
else:
    print("No token scores collected (RUN_TOKEN_SCORING False or all scoring passes failed).")

### 8b. Diagnosing and repairing a low score/generation agreement

Protocol 7.2 requires the scored `argmax` to reproduce the model's own free-generation decision
on ≥ 99.5% of cases. A shortfall is **not** noise to be tolerated — it means the scoring pass and
the generation pass are asking the model two different questions.

The usual cause is a **format mismatch**. The scoring candidates are the canonical
`{"covid_positive":"Yes"}`, but a model may actually emit `{"covid_positive": "Yes"}` (space
after the colon), or wrap it in a ```json fence, or prefix a short preamble. Teacher-forcing a
string the model would never have produced measures the likelihood of an off-distribution
continuation, and the Yes/No ratio drifts.

The repair is to build the candidates from the model's **own emitted text**: take the generated
output, substitute `Yes`/`No` into the value position, and score those two strings. The scoring
path then matches the generation path exactly, by construction.

The cell below first *characterises* the disagreements (are they concentrated at small
log-likelihood margins, or spread out? does the emitted string differ from the canonical one?),
then — for arms listed in `RESCORE_ARMS` — re-scores using aligned candidates. Cached
generations are reused, so only the scoring forward passes are repeated.


In [ ]:
# ---- Diagnose ----------------------------------------------------------------------------
disagreement_report = []
if len(score_frame):
    predictions_by_key = {
        (row["arm"], row["image_key"]): row for row in completed.values()
        if row["task"] == "covid_classification"
    }
    for arm in sorted(score_frame["arm"].unique()):
        subset = score_frame[(score_frame["arm"] == arm)
                             & score_frame["generated_decision"].notna()].copy()
        if not len(subset):
            continue
        subset["agrees"] = subset["scored_decision"] == subset["generated_decision"]
        subset["margin"] = (subset["logL_yes"] - subset["logL_no"]).abs()
        bad = subset[~subset["agrees"]]

        canonical_yes = covid_candidate_json("Yes")
        canonical_no = covid_candidate_json("No")
        n_exact_format, n_fenced, n_preamble, examples = 0, 0, 0, []
        for _, row in bad.head(200).iterrows():
            source = predictions_by_key.get((arm, row["image_key"]))
            raw = (source or {}).get("raw_output", "") or ""
            emitted = raw.strip()
            if emitted.startswith("```"):
                n_fenced += 1
            if emitted and not emitted.startswith("{"):
                n_preamble += 1
            if canonical_yes in raw or canonical_no in raw:
                n_exact_format += 1
            if len(examples) < 3:
                examples.append(raw[:160])

        disagreement_report.append({
            "arm": arm, "n_scored": len(subset),
            "agreement": round(float(subset["agrees"].mean()), 5),
            "n_disagree": int(len(bad)),
            "median_margin_agree": round(float(subset[subset["agrees"]]["margin"].median()), 3)
            if subset["agrees"].any() else None,
            "median_margin_disagree": round(float(bad["margin"].median()), 3) if len(bad) else None,
            "disagree_with_margin_lt_1": int((bad["margin"] < 1.0).sum()) if len(bad) else 0,
            "emitted_canonical_format": n_exact_format,
            "emitted_code_fence": n_fenced,
            "emitted_preamble_before_brace": n_preamble,
            "examples": examples,
        })

    report_frame = pd.DataFrame(disagreement_report)
    report_frame.drop(columns=["examples"]).to_csv(
        NB07_DIR / "score_disagreement_diagnosis.csv", index=False)
    print(report_frame.drop(columns=["examples"]).to_string(index=False))
    print()
    for entry in disagreement_report:
        if entry["agreement"] >= SCORE_AGREEMENT_THRESHOLD:
            continue
        print(f"--- {entry['arm']}: agreement {entry['agreement']:.4f}, "
              f"{entry['n_disagree']} disagreements")
        near_tie = entry["disagree_with_margin_lt_1"] / max(entry["n_disagree"], 1)
        print(f"    {near_tie:.0%} of disagreements have |log-likelihood margin| < 1, i.e. the "
              "model is genuinely undecided there.")
        print(f"    emitted canonical format: {entry['emitted_canonical_format']}, "
              f"code fence: {entry['emitted_code_fence']}, "
              f"preamble before '{{': {entry['emitted_preamble_before_brace']}")
        if entry["emitted_canonical_format"] < entry["n_disagree"] * 0.5:
            print("    DIAGNOSIS: the model rarely emits the canonical candidate string, so the")
            print("    scoring pass is teacher-forcing an off-distribution continuation.")
            print(f"    FIX: add '{entry['arm']}' to RESCORE_ARMS and re-run this cell.")
        else:
            print("    DIAGNOSIS: format matches, so the disagreements are near-ties rather")
            print("    than a scoring bug. Aligned rescoring will help little; the arm's score")
            print("    is weakly informative and will be quarantined by the gate.")
        for example in entry["examples"]:
            print(f"      raw: {example!r}")
        print()


# ---- Repair: candidates built from the model's own emitted format --------------------------
def aligned_candidates(raw_output):
    """
    Return (yes_text, no_text) matching the model's own emitted formatting, or None if the
    output cannot be aligned. Preserves any preamble, fence, and whitespace.
    """
    if not raw_output:
        return None
    match = re.search(r'("covid_positive"\s*:\s*")(Yes|No|yes|no|YES|NO)(")', raw_output)
    if not match:
        return None
    end = raw_output.find("}", match.end())
    if end < 0:
        return None
    prefix = raw_output[:match.end(1)]
    suffix = raw_output[match.start(3):end + 1]
    return prefix + "Yes" + suffix, prefix + "No" + suffix


rescore_records = []
if RESCORE_ARMS and (FORCE_RESCORE or not stage_complete(
        "rescore", fingerprint(sorted(RESCORE_ARMS), GENERATION_FINGERPRINT),
        [NB07_DIR / "score_records_aligned.jsonl"])):
    ALIGNED_PATH = NB07_DIR / "score_records_aligned.jsonl"
    done = {(row["arm"], row["image_key"])
            for row in cm.read_jsonl(ALIGNED_PATH)} if ALIGNED_PATH.is_file() else set()
    rescore_records = cm.read_jsonl(ALIGNED_PATH) if ALIGNED_PATH.is_file() else []

    for arm in RESCORE_ARMS:
        spec = MODEL_ARMS[arm]
        targets = [row for row in completed.values()
                   if row["arm"] == arm and row["task"] == "covid_classification"
                   and (arm, row["image_key"]) not in done]
        if not targets:
            print(f"{arm}: aligned rescoring already complete.")
            continue
        print(f"{arm}: aligned rescoring {len(targets):,} images "
              "(reusing cached generations; scoring passes only)")
        model, processor_arm, revision = load_vlm(spec)
        saved_processor = globals()["processor"] if "processor" in globals() else None
        try:
            started = time.perf_counter()
            for position, row in enumerate(targets, start=1):
                candidates = aligned_candidates(row.get("raw_output") or "")
                if candidates is None:
                    continue
                image_row = work[work["image_key"] == row["image_key"]]
                if not len(image_row):
                    continue
                image_path = image_row.iloc[0]["v0_image"]
                system_prompt = templates["covid_classification"]["system"]
                user_prompt = templates["covid_classification"]["user"]
                try:
                    yes_ll, _ = sequence_log_likelihood(
                        model, processor_arm, spec, image_path, system_prompt, user_prompt,
                        candidates[0])
                    no_ll, _ = sequence_log_likelihood(
                        model, processor_arm, spec, image_path, system_prompt, user_prompt,
                        candidates[1])
                except Exception as exc:
                    print(f"    {row['image_key']}: {type(exc).__name__}: {exc}")
                    continue
                if math.isnan(yes_ll) or math.isnan(no_ll):
                    continue
                maximum = max(yes_ll, no_ll)
                probability = (math.exp(yes_ll - maximum)
                               / (math.exp(yes_ll - maximum) + math.exp(no_ll - maximum)))
                entry = {
                    "arm": arm, "image_key": row["image_key"], "covid_score": probability,
                    "logL_yes": yes_ll, "logL_no": no_ll,
                    "generated_decision": row.get("covid_pred"),
                    "scored_decision": "Yes" if probability >= 0.5 else "No",
                    "gt_covid": row.get("gt_covid"), "alignment": "model_emitted_format",
                }
                cm.append_jsonl(ALIGNED_PATH, entry)
                rescore_records.append(entry)
                if position % 100 == 0:
                    rate = position / max(time.perf_counter() - started, 1e-6)
                    print(f"    {position}/{len(targets)} ({rate:.2f} img/s)")
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if rescore_records:
        mark_stage("rescore", fingerprint(sorted(RESCORE_ARMS), GENERATION_FINGERPRINT),
                   {"n_rescored": len(rescore_records), "arms": RESCORE_ARMS})
elif RESCORE_ARMS:
    ALIGNED_PATH = NB07_DIR / "score_records_aligned.jsonl"
    rescore_records = cm.read_jsonl(ALIGNED_PATH) if ALIGNED_PATH.is_file() else []
    print(f"SKIPPING section 8b: aligned rescoring already complete "
          f"({len(rescore_records):,} records). Set FORCE_RESCORE = True to redo.")
else:
    print("RESCORE_ARMS is empty; no aligned rescoring requested.")

# Aligned scores override the canonical ones for the arms that were rescored.
if rescore_records:
    aligned_by_key = {(row["arm"], row["image_key"]): row for row in rescore_records}
    replaced = 0
    for record in score_records:
        key = (record["arm"], record["image_key"])
        if key in aligned_by_key:
            record.update({k: aligned_by_key[key][k] for k in
                           ["covid_score", "logL_yes", "logL_no", "scored_decision"]})
            record["alignment"] = "model_emitted_format"
            replaced += 1
    for key, row in completed.items():
        arm, image_key, task = key
        if task == "covid_classification" and (arm, image_key) in aligned_by_key:
            row["covid_score"] = aligned_by_key[(arm, image_key)]["covid_score"]
    score_frame = pd.DataFrame(score_records)
    print()
    print(f"Applied {replaced:,} aligned scores; re-run section 8 to refresh the validation "
          "table.")
    for arm in RESCORE_ARMS:
        subset = score_frame[(score_frame["arm"] == arm)
                             & score_frame["generated_decision"].notna()]
        if len(subset):
            agreement = float((subset["scored_decision"] == subset["generated_decision"]).mean())
            verdict = "PASS" if agreement >= SCORE_AGREEMENT_THRESHOLD else "still failing"
            print(f"  {arm}: aligned agreement = {agreement:.5f}  [{verdict}]")


## 9. Metrics, output integrity, and the E6-Q measurement

In [ ]:
def rows_for(arm, task=None, internal_only=True):
    selected = []
    for row in predictions:
        if row["arm"] != arm:
            continue
        if task is not None and row["task"] != task:
            continue
        if internal_only and row.get("held_out_fold") is None:
            continue
        selected.append(row)
    return selected


def merge_tasks(arm, internal_only=True):
    """One row per image combining the COVID and mRALE task outputs, for joint metrics."""
    merged = {}
    for row in rows_for(arm, internal_only=internal_only):
        entry = merged.setdefault(row["image_key"], {
            key: row.get(key) for key in
            ["image_key", "cohort", "subcohort", "filename", "held_out_fold",
             "gt_covid", "gt_mrale_total", "gt_mrale_right", "gt_mrale_left",
             "gt_extent_right", "gt_density_right", "gt_extent_left", "gt_density_left"]
        })
        if row["task"] == "covid_classification":
            entry["covid_pred"] = row.get("covid_pred")
            entry["covid_score"] = row.get("covid_score")
            entry["covid_valid"] = row.get("valid")
        else:
            for key in ["mrale_total", "mrale_right", "mrale_left", "extent_right",
                        "density_right", "extent_left", "density_left",
                        "mrale_total_expected", "mrale_uncertainty"]:
                entry[key] = row.get(key)
            entry["mrale_valid"] = row.get("valid")
        entry["valid"] = bool(entry.get("covid_valid", True)) and bool(entry.get("mrale_valid", True))
        entry["parse_error"] = entry.get("parse_error") or row.get("parse_error")
        entry["seconds"] = (entry.get("seconds") or 0) + (row.get("seconds") or 0)
    return list(merged.values())


METRICS_FINGERPRINT = fingerprint(
    GENERATION_FINGERPRINT, len(predictions), len(score_records),
    sorted(RESCORE_ARMS), SCORE_AGREEMENT_THRESHOLD)
METRICS_FILES = [NB07_DIR / "arm_summary.csv", NB07_DIR / "output_integrity.csv"]

SKIP_METRICS = (not FORCE_RECOMPUTE_METRICS
                and stage_complete("metrics", METRICS_FINGERPRINT, METRICS_FILES))
if SKIP_METRICS:
    summary = pd.read_csv(NB07_DIR / "arm_summary.csv")
    integrity = pd.read_csv(NB07_DIR / "output_integrity.csv")
    print("SKIPPING SECTION 9: metrics already computed for these predictions; reloaded from "
          "disk. Set FORCE_RECOMPUTE_METRICS = True to redo.")
    print()
    print(summary[["arm", "mrale_mae_pooled", "covid_auroc_pooled",
                   "covid_balanced_accuracy", "mrale_coverage"]].to_string(index=False))

summary_rows = []
integrity_rows = []

for arm in ([] if SKIP_METRICS else RUN_ARMS):
    merged = merge_tasks(arm)
    if not merged:
        continue
    pooled = evaluate_arm(merged, arm)
    covid, mrale = pooled.get("covid", {}), pooled.get("mrale", {})

    per_fold = {}
    for fold in range(N_FOLDS):
        fold_rows = [row for row in merged if row.get("held_out_fold") == fold]
        if fold_rows:
            per_fold[fold] = evaluate_arm(fold_rows, f"{arm}/fold{fold}")
    aggregate = cm.aggregate_over_folds(per_fold) if per_fold else []
    if aggregate:
        pd.DataFrame(aggregate).to_csv(
            NB07_DIR / f"cross_validation_aggregate_95ci_{arm}.csv", index=False)

    def fold_ci(metric):
        match = [row for row in aggregate if row["metric"] == metric]
        return ((match[0]["mean"], match[0]["ci95_lower"], match[0]["ci95_upper"])
                if match else (None, None, None))

    mae_mean, mae_low, mae_high = fold_ci("mrale.mae")
    auroc_mean, auroc_low, auroc_high = fold_ci("covid.auroc")
    validation_entry = next((row for row in validation_rows if row["arm"] == arm), {})

    summary_rows.append(OrderedDict([
        ("arm", arm),
        ("model_id", MODEL_ARMS[arm]["model_id"]),
        ("agent", MODEL_ARMS[arm]["agent"]),
        ("adaptation", "zero-shot"),
        ("n_images", len(merged)),
        ("mrale_mae_pooled", round(mrale.get("mae", float("nan")), 3)),
        ("mrale_mae_ci95", None if mae_mean is None else f"[{mae_low:.3f}, {mae_high:.3f}]"),
        ("covid_auroc_pooled", round(covid.get("auroc", float("nan")), 4)),
        ("covid_auroc_ci95", None if auroc_mean is None else f"[{auroc_low:.4f}, {auroc_high:.4f}]"),
        ("covid_auprc", round(covid.get("auprc", float("nan")), 4)),
        ("covid_balanced_accuracy", round(covid.get("balanced_accuracy", float("nan")), 4)),
        ("covid_sensitivity", round(covid.get("sensitivity", float("nan")), 4)),
        ("covid_specificity", round(covid.get("specificity", float("nan")), 4)),
        ("covid_f1", round(covid.get("f1", float("nan")), 4)),
        ("covid_mcc", round(covid.get("mcc", float("nan")), 4)),
        ("covid_brier", round(covid.get("brier", float("nan")), 4)),
        ("covid_ece", round(covid.get("ece", float("nan")), 4)),
        ("mrale_rmse", round(mrale.get("rmse", float("nan")), 3)),
        ("mrale_qwk", round(mrale.get("qwk", float("nan")), 4)),
        ("mrale_spearman", round(mrale.get("spearman_rho", float("nan")), 4)),
        ("mrale_within1", round(mrale.get("within1_accuracy", float("nan")), 4)),
        ("mrale_coverage", round(mrale.get("coverage", float("nan")), 4)),
        ("mrale_formula_consistency", round(mrale.get("formula_consistency", float("nan")), 4)),
        ("mae_band_none", round(mrale.get("mae_band_none", float("nan")), 3)),
        ("mae_band_mild", round(mrale.get("mae_band_mild", float("nan")), 3)),
        ("mae_band_moderate", round(mrale.get("mae_band_moderate", float("nan")), 3)),
        ("mae_band_severe", round(mrale.get("mae_band_severe", float("nan")), 3)),
        ("score_generation_agreement", validation_entry.get("score_generation_agreement")),
    ]))

    for task in TASKS:
        task_rows = rows_for(arm, task=task, internal_only=False)
        if not task_rows:
            continue
        objects = [row.get("extra", {}).get("n_json_objects", 0) for row in task_rows]
        tokens = [row.get("extra", {}).get("n_completion_tokens", 0) for row in task_rows]
        integrity_rows.append({
            "arm": arm, "task": task, "n": len(task_rows),
            "valid_rate": round(float(np.mean([bool(r.get("valid")) for r in task_rows])), 4),
            # E6-Q: >1 means the model repeated its answer.
            "duplicate_object_rate": round(
                float(np.mean([count > 1 for count in objects])), 4),
            "mean_json_objects": round(float(np.mean(objects)), 3),
            "median_completion_tokens": int(np.median(tokens)) if tokens else None,
            "median_seconds": round(float(np.median(
                [r.get("seconds") or 0 for r in task_rows])), 3),
            "p95_seconds": round(float(np.percentile(
                [r.get("seconds") or 0 for r in task_rows], 95)), 3),
            "n_parse_errors": int(sum(1 for r in task_rows if r.get("parse_error"))),
        })

if not SKIP_METRICS:
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(NB07_DIR / "arm_summary.csv", index=False)
    integrity = pd.DataFrame(integrity_rows)
    integrity.to_csv(NB07_DIR / "output_integrity.csv", index=False)
    mark_stage("metrics", METRICS_FINGERPRINT,
               {"arms": RUN_ARMS, "n_predictions": len(predictions)})

pd.set_option("display.width", 220)
print(summary[["arm", "mrale_mae_pooled", "mrale_mae_ci95", "covid_auroc_pooled",
               "covid_auroc_ci95", "covid_balanced_accuracy", "covid_specificity",
               "mrale_coverage"]].to_string(index=False))
print()
print("Output integrity (E6-Q check):")
print(integrity.to_string(index=False))
print()

# ---- Zero-coverage triage -----------------------------------------------------------------
# Coverage of exactly 0.000 across thousands of images is a FORMAT BUG, not a weak model.
# NV-Reason in particular is chain-of-thought trained and may narrate before (or instead of)
# emitting JSON, in which case the balanced-brace stop criterion can halt inside the prose.
for _, row in summary.iterrows():
    coverage = row.get("mrale_coverage")
    if coverage is None or (isinstance(coverage, float) and math.isnan(coverage)):
        continue
    if float(coverage) > 0.0:
        continue
    arm = row["arm"]
    print("=" * 78)
    print(f"{arm}: mRALE coverage is EXACTLY 0.000 -- no output parsed, on any image.")
    print("This is a format mismatch, not a result. Raw outputs below:")
    samples = [r for r in predictions
               if r["arm"] == arm and r["task"] == "mrale_prediction"][:3]
    for sample in samples:
        print(f"  parse_error : {sample.get('parse_error')}")
        print(f"  n_tokens    : {sample.get('extra', {}).get('n_completion_tokens')}")
        print(f"  raw_output  : {(sample.get('raw_output') or '')[:400]!r}")
        print()
    print("  Likely causes and fixes:")
    print("   1. The model narrates before emitting JSON, and USE_JSON_STOP_CRITERIA halted")
    print("      inside the prose at a brace that was not the answer object. Try")
    print("      USE_JSON_STOP_CRITERIA = False for this arm and raise max_new_tokens.")
    print("   2. It never emits JSON at all under this prompt. A reasoning-tuned model often")
    print("      needs the schema restated as an explicit final instruction; add an arm-level")
    print("      prompt suffix rather than changing the shared template, so E0a/E0b stay")
    print("      comparable.")
    print("   3. max_new_tokens is too small for a chain-of-thought preamble plus the JSON")
    print(f"      (currently {MAX_NEW_TOKENS['mrale_prediction']}).")
    print("  Until fixed, this arm's mRALE numbers are uninformative: its penalised MAE is")
    print("  exactly the 24-point invalid penalty and says nothing about its ability.")
    print()
if len(integrity) and integrity["duplicate_object_rate"].max() > 0.01:
    print("Duplicate JSON objects are still being emitted. The JSON stop criteria did not")
    print("fire for some arm; inspect raw_output. Metrics remain valid (the first object is")
    print("parsed) but token counts, latency, and any E6-C self-consistency arm would be")
    print("measuring repetition.")
else:
    print("Duplicate-object rate is at or near zero: the E6-Q stop-criteria fix is working.")

### 9b. NV-Reason: output-format incompatibility and severity-dependent abstention

The zero-coverage diagnosis was wrong twice over, and the trace file settles it. The image **is**
bound (`placeholder=True`, `pixel_values;image_grid_thw` present) and the model **does** read the
radiograph:

> `<think> We'll begin with the quality assessment of this PA chest x-ray. As you can see, the lung
> fields are well visualized, and the scapulae do not superimpose over the lung parenchyma...
> I don't see any medical devices, such as central venous catheters, EKG leads, or chest tubes...`

Two separate findings, both reportable rather than fixable-and-forget.

**1. Its output space is not JSON, and not mRALE.** Every one of the 1,262 substantive traces ends
in `</think>\n<answer> Atelectasis, Lung Opacity </answer>`. Zero contain a `{`. NV-Reason was
RL-trained to emit a fixed CheXpert-style finding list, so no prompt suffix will coax mRALE
components out of it — the quantity simply is not in its output vocabulary. `mrale_usable=False`
for this arm is therefore **correct and permanent under zero-shot**, and should be reported as
*"output space incompatible with the mRALE rubric"*, not as a parsing failure. Reporting 24.0 would
be doubly wrong.

**2. It abstains more often the sicker the patient.** Refusal ("Please provide a frontal chest
x-ray image") rises monotonically with annotated severity:

| mRALE | refusal rate |
| --- | --- |
| 0 | 38.5% |
| 1–10 | 45.9% |
| 11–18 | 67.1% |
| 19–24 | **86.1%** |

The obvious confound is image appearance — a whiteout film is bright and flat, so maybe the model
rejects low-contrast images. **The data says no.** Holding mean intensity fixed in the 80–120 band,
refusal still runs 27.3% → 39.7% → 64.2% → 84.7% across the severity bands, and the same gradient
appears in the 120–160 band. Brightness, contrast, and image dimensions do not explain it;
severity does, independently.

The mechanism is visible in the traces: the model's trained protocol *opens* with a quality
assessment, and on a heavily opacified film it appears to conclude the study is inadequate and ask
for another. A reasoning-tuned CXR model that abstains hardest exactly where severity assessment
matters most is a genuine, citable robustness finding — and a better contribution from this arm
than a forced accuracy number would have been.

This cell quantifies both, writes the artifacts, and leaves the arm honestly labelled.


In [ ]:
NVREASON_ARM = "E0c_nvreason"
ANSWER_TAG = re.compile(r"<answer>(.*?)</answer>", re.S | re.I)
REFUSAL_PATTERN = re.compile(r"please provide|provide a frontal|cannot see|no image", re.I)


def is_abstention(text):
    return bool(REFUSAL_PATTERN.search(text or "")) and len(text or "") < 200


nvreason_findings = {}
abstention_rows = []

trace_path = NB07_DIR / "nvreason_traces.jsonl"
if trace_path.is_file():
    traces = cm.read_jsonl(trace_path)
    print(f"NV-Reason traces: {len(traces):,}")

    manifest_path = NB01_DIR / "midrc_manifest.csv"
    appearance = {}
    if manifest_path.is_file():
        manifest = pd.read_csv(manifest_path)
        for _, row in manifest.iterrows():
            appearance[str(row["filename"])] = {
                "mean_intensity": row.get("mean_intensity"),
                "stddev_intensity": row.get("stddev_intensity"),
                "min_side": min(row.get("width", 0) or 0, row.get("height", 0) or 0),
            }

    for trace in traces:
        text = trace.get("trace") or ""
        filename = str(trace.get("image_key", "")).split("::")[-1]
        labels = []
        match = ANSWER_TAG.search(text)
        if match:
            labels = [item.strip() for item in match.group(1).split(",") if item.strip()]
        abstention_rows.append({
            "image_key": trace.get("image_key"),
            "gt_mrale_total": trace.get("gt_mrale_total"),
            "gt_covid": trace.get("gt_covid"),
            "severity_band": cm.severity_band(trace.get("gt_mrale_total")),
            "abstained": is_abstention(text),
            "trace_chars": len(text),
            "has_think_block": "<think>" in text,
            "has_answer_tag": bool(match),
            "n_labels": len(labels),
            "labels": ";".join(labels),
            "contains_json_brace": "{" in text,
            **appearance.get(filename, {}),
        })
        if labels:
            nvreason_findings[trace.get("image_key")] = labels

    frame = pd.DataFrame(abstention_rows)
    frame.to_csv(NB07_DIR / "nvreason_abstention.csv", index=False)

    n = len(frame)
    n_abstain = int(frame["abstained"].sum())
    print(f"  abstention rate      : {n_abstain}/{n} = {n_abstain / n:.1%}")
    print(f"  traces with <answer> : {int(frame['has_answer_tag'].sum())}")
    print(f"  traces with a '{{'    : {int(frame['contains_json_brace'].sum())}")
    print()
    print("  FINDING 1: the output space is a CheXpert-style finding list, not JSON and not")
    print("  mRALE. No prompt change can extract a quantity the model does not emit, so")
    print("  mrale_usable=False is correct and permanent for this arm under zero-shot.")
    print()

    print("  FINDING 2: abstention rises with annotated severity")
    by_band = frame.groupby("severity_band")["abstained"].agg(["sum", "count"])
    for band in ["none", "mild", "moderate", "severe"]:
        if band in by_band.index:
            hit, total = int(by_band.loc[band, "sum"]), int(by_band.loc[band, "count"])
            print(f"    {band:<10} {hit:>4}/{total:<5} = {hit / total:6.1%}")

    # Is it appearance or severity? Hold intensity fixed and look again.
    if "mean_intensity" in frame.columns and frame["mean_intensity"].notna().any():
        print()
        print("  Confound check -- refusal within FIXED mean-intensity bands:")
        for low, high in [(80, 120), (120, 160)]:
            band_subset = frame[(frame["mean_intensity"] >= low)
                                & (frame["mean_intensity"] < high)]
            if len(band_subset) < 60:
                continue
            print(f"    intensity {low}-{high} (n={len(band_subset)}):")
            inner = band_subset.groupby("severity_band")["abstained"].agg(["sum", "count"])
            for band in ["none", "mild", "moderate", "severe"]:
                if band in inner.index and int(inner.loc[band, "count"]) >= 15:
                    hit = int(inner.loc[band, "sum"]); total = int(inner.loc[band, "count"])
                    print(f"        {band:<10} {hit:>4}/{total:<5} = {hit / total:6.1%}")
        print()
        print("  The severity gradient survives conditioning on brightness, so it is not a")
        print("  low-contrast artefact. Report it as a property of the model.")

    labels_counter = Counter(label for labels in nvreason_findings.values() for label in labels)
    if labels_counter:
        pd.DataFrame(labels_counter.most_common(),
                     columns=["finding", "n"]).to_csv(
            NB07_DIR / "nvreason_label_distribution.csv", index=False)
        print()
        print("  Finding labels emitted (its actual output vocabulary):")
        for label, count in labels_counter.most_common(12):
            print(f"    {label:<34} {count:>5}")

    cm.write_json(NB07_DIR / "nvreason_finding.json", {
        "arm": NVREASON_ARM,
        "image_binding": "CONFIRMED WORKING (placeholder present, pixel_values + "
                         "image_grid_thw produced, traces describe the radiograph)",
        "finding_1_output_space": {
            "format": "<think>...</think><answer>label, label</answer>",
            "n_traces_with_answer_tag": int(frame["has_answer_tag"].sum()),
            "n_traces_with_json": int(frame["contains_json_brace"].sum()),
            "conclusion": (
                "Output space is a fixed CheXpert-style finding list. mRALE components are not "
                "in its output vocabulary, so zero-shot mRALE is impossible for this arm by "
                "construction, not by prompt design. Report as 'output space incompatible with "
                "the mRALE rubric'."
            ),
        },
        "finding_2_severity_dependent_abstention": {
            "overall_rate": round(n_abstain / n, 4),
            "by_band": {band: round(int(by_band.loc[band, "sum"])
                                    / int(by_band.loc[band, "count"]), 4)
                        for band in by_band.index},
            "confound_tested": "mean intensity, pixel std-dev, image dimensions",
            "conclusion": (
                "Abstention rises monotonically with annotated severity and survives "
                "conditioning on image brightness, so it is a property of the model rather "
                "than a low-contrast artefact. A reasoning-tuned CXR model that abstains "
                "hardest where severity assessment matters most is a reportable robustness "
                "limitation."
            ),
        },
        "finding_3_prompt_mode_interaction": {
            "refusal_schema_only_prompt": "~98% (covid), ~81% (mrale)",
            "refusal_with_reasoning_instruction": "~54%",
            "conclusion": (
                "Demanding JSON suppresses the model's reasoning mode: the same image and the "
                "same weights refuse far more often when the prompt asks only for a schema. "
                "Any comparison of this model against the others is therefore confounded by "
                "output-format compliance, not just by capability."
            ),
        },
        "reporting_guidance": [
            "Do NOT report a 24.0 mRALE MAE for this arm; omit the mRALE row and state why.",
            "Its covid_score came from teacher-forcing JSON candidates the model never emits, "
            "so it is meaningless -- score_usable=False is correct.",
            "A legitimate score IS obtainable by scoring in its own format: "
            "'<answer> Lung Opacity </answer>' against '<answer> No Finding </answer>'. "
            "That is a fair contrast and would give this arm a usable AUROC.",
            "RQ2 must be restated: NV-Reason cannot be compared on mRALE, so RQ2 becomes a "
            "detection-and-abstention question, not a severity question.",
        ],
    })
    print()
    print("Wrote nvreason_finding.json, nvreason_abstention.csv, "
          "nvreason_label_distribution.csv")
else:
    print(f"{trace_path.name} not found; run section 7 with capture_traces enabled for "
          "NV-Reason to reproduce this analysis.")


### RQ1 and RQ2 read-out

- **RQ1 (medical specialization at matched scale):** `E0a_medgemma` vs `E0b_qwen35`, both 4B,
  identical prompts, identical folds. Any difference is attributable to pretraining.
- **RQ2 (reasoning-tuned pretraining):** `E0c_nvreason` at 3B. Note the scale is *not* matched,
  so a favourable result is confounded with architecture and data as well as reasoning tuning.
  Say so rather than over-claiming.

Zero-shot coverage matters as much as accuracy here. A model with mRALE coverage of 0.4 has a
flattering valid-only MAE and a terrible penalised MAE, and the penalised figure is the
pre-registered primary endpoint P1. Report them together, always.

## 10. Run configuration and gate

In [ ]:
cm.write_json(NB07_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "07_zeroshot_vlm_agents.ipynb",
    "protocol_experiments": ["E0a", "E0b", "E0c"],
    "enabling_fix": "Section 7.2 token-probability scoring",
    "seed": SEED,
    "arms": {arm: {**MODEL_ARMS[arm],
                   "revision": MODEL_REVISIONS.get(MODEL_ARMS[arm]["model_id"])}
             for arm in RUN_ARMS},
    "decoding": {
        "do_sample": False, "max_new_tokens": MAX_NEW_TOKENS,
        "json_stop_criteria": USE_JSON_STOP_CRITERIA,
        "note": "Greedy everywhere. Experiment family E6 owns the temperature/top-p sweep.",
    },
    "scoring": {
        "method": "exact teacher-forced sequence log-likelihood over candidate completions",
        "covid_candidates": [covid_candidate_json(v) for v in COVID_CANDIDATES],
        "length_normalisation": False,
        "length_normalisation_rationale": (
            "Both candidates are the same JSON with one word substituted, so a length "
            "correction would add an arbitrary constant without changing the ranking."
        ),
        "mrale_field_scoring": SCORE_MRALE_FIELDS,
        "mrale_field_method": (
            "Teacher-force the model's own generated JSON, locate the numeric field token "
            "positions, and read the distribution over single-token digits allowed at each "
            "position (extent 0-4, density 0-3)."
        ),
        "validation_threshold": SCORE_AGREEMENT_THRESHOLD,
        "agreement_definition": (
            "Reported over all scored cases AND over decisive cases only "
            f"(|logL_yes - logL_no| >= {AGREEMENT_DECISIVE_MARGIN} nats). The gate uses the "
            "decisive figure: the claim that matters is that the score reflects the model's "
            "preference where it has one, and on a near-tie the greedy decode and the "
            "likelihood ranking may differ without either being wrong. The near-tie fraction "
            "is reported so the exclusion is visible rather than hidden."
        ),
        "decisive_margin_nats": AGREEMENT_DECISIVE_MARGIN,
        "gate_on_decisive_agreement": GATE_ON_DECISIVE_AGREEMENT,
        "validation": validation_rows,
        "quarantine_policy": (
            "An arm failing the agreement threshold is marked score_usable=False in "
            "score_usability.json rather than aborting the run. NB 17/18 must exclude it from "
            "ROC, PR, DeLong, and calibration; its hard-decision metrics stay valid."
        ),
        "rescored_arms": RESCORE_ARMS,
        "aligned_candidate_method": (
            "Candidates rebuilt from the model's own emitted string so the scoring pass "
            "matches the generation pass by construction."
        ),
    },
    "constraints": {
        "mrale_total": "constrained to extent_right*density_right + extent_left*density_left",
        "reported_total_kept_separately": True,
    },
    "prompts": {task: templates[task] for task in TASKS},
    "prompt_source": str(templates_path),
    "arm_prompt_suffix": ARM_PROMPT_SUFFIX,
    "image_binding": {
        "preflight_run": RUN_IMAGE_PREFLIGHT,
        "template_variant_used": IMAGE_TEMPLATE_VARIANT,
        "preflight": preflight_rows if "preflight_rows" in dir() else None,
        "note": (
            "A valueless {'type': 'image'} entry renders no vision token in some "
            "chat templates (Qwen2.5-VL derivatives), so the model never sees the "
            "image and returns fluent text with zero parseable JSON. render_prompt "
            "selects the first message variant that yields a placeholder."
        ),
    },
    "data": {"n_images": int(len(work)),
             "internal": int((~work["is_external"]).sum()),
             "external": int(work["is_external"].sum())},
    "max_images_per_arm": MAX_IMAGES_PER_ARM,
})

failures, warnings = [], []

# ---- Self-heal ---------------------------------------------------------------------------
# This cell must not fail merely because section 8 was not re-executed in this kernel. Every
# input it needs is already on disk, so rebuild rather than complain.
def _validation_from_records(records):
    frame = pd.DataFrame(records)
    if not len(frame) or "generated_decision" not in frame.columns:
        return [], frame
    frame = frame[frame["generated_decision"].notna()]
    rows = []
    for arm in sorted(frame["arm"].unique()):
        subset = frame[frame["arm"] == arm]
        if not len(subset):
            continue
        agreement = float((subset["scored_decision"] == subset["generated_decision"]).mean())
        rows.append({
            "arm": arm, "n_scored": len(subset),
            "score_generation_agreement": round(agreement, 5),
            "mean_abs_logL_margin": round(
                float((subset["logL_yes"] - subset["logL_no"]).abs().mean()), 3)
            if subset["logL_yes"].notna().any() else None,
            "mean_score": round(float(subset["covid_score"].mean()), 4),
            "score_std": round(float(subset["covid_score"].std()), 4),
            "meets_protocol_7_2": bool(agreement >= SCORE_AGREEMENT_THRESHOLD),
        })
    return rows, frame


# Precedence matters. Section 8 computes agreement with the parsed decision in scope; a
# rebuild from disk is a reconstruction. If section 8 already wrote its table, TRUST THAT --
# a rebuild that silently disagrees with it would change an arm's verdict for no good reason.
VALIDATION_CSV = NB07_DIR / "score_validation.csv"
validation_source = "section 8 (this kernel)" if validation_rows else None

if RUN_TOKEN_SCORING and not validation_rows and VALIDATION_CSV.is_file():
    saved = pd.read_csv(VALIDATION_CSV)
    if len(saved) and "score_generation_agreement" in saved.columns:
        validation_rows = saved.to_dict(orient="records")
        for row in validation_rows:
            row["meets_protocol_7_2"] = bool(
                float(row["score_generation_agreement"]) >= SCORE_AGREEMENT_THRESHOLD)
        validation_source = f"score_validation.csv ({len(validation_rows)} arms)"
        print(f"Loaded the validation table written by section 8: {validation_source}")

if RUN_TOKEN_SCORING and not validation_rows:
    validation_source = "rebuilt from predictions_zeroshot.jsonl"
    recovered = []
    for row in predictions:
        if row.get("task") != "covid_classification" or row.get("covid_score") is None:
            continue
        candidates = (row.get("extra") or {}).get("candidate_log_likelihoods") or {}
        probability = row["covid_score"]
        recovered.append({
            "arm": row["arm"], "image_key": row["image_key"], "covid_score": probability,
            "logL_yes": (candidates.get("Yes") or {}).get("log_likelihood"),
            "logL_no": (candidates.get("No") or {}).get("log_likelihood"),
            "generated_decision": row.get("covid_pred"),
            "scored_decision": "Yes" if probability >= 0.5 else "No",
            "gt_covid": row.get("gt_covid"), "recovered_from_predictions": True,
        })
    aligned_path = NB07_DIR / "score_records_aligned.jsonl"
    if aligned_path.is_file():
        aligned = {(r["arm"], r["image_key"]): r for r in cm.read_jsonl(aligned_path)}
        for record in recovered:
            key = (record["arm"], record["image_key"])
            if key in aligned:
                record.update({k: aligned[key][k] for k in
                               ["covid_score", "logL_yes", "logL_no", "scored_decision"]})
                record["alignment"] = "model_emitted_format"
    if recovered:
        score_records = recovered
        validation_rows, score_frame = _validation_from_records(recovered)
        # Separate file: overwriting score_validation.csv would replace section 8's
        # authoritative table with a reconstruction, and any small difference between them
        # would silently change an arm's verdict.
        pd.DataFrame(validation_rows).to_csv(
            NB07_DIR / "score_validation_rebuilt.csv", index=False)
        print(f"SELF-HEAL: rebuilt {len(recovered):,} score records and "
              f"{len(validation_rows)} validation rows from predictions_zeroshot.jsonl "
              "(section 8 had not run in this kernel). Nothing was recomputed.")
        for row in validation_rows:
            print(f"  {row['arm']}: agreement={row['score_generation_agreement']:.5f} "
                  f"n={row['n_scored']} "
                  f"{'PASS' if row['meets_protocol_7_2'] else 'quarantine'}")
        print()

# Per-arm score availability. Printed unconditionally, because "arm absent from the
# validation table" and "arm present but failing" are different problems with different fixes,
# and conflating them is what produced a misleading gate message.
availability_rows = []
for arm in RUN_ARMS:
    arm_covid = [row for row in predictions
                 if row["arm"] == arm and row.get("task") == "covid_classification"]
    availability_rows.append({
        "arm": arm,
        "n_covid_rows": len(arm_covid),
        "n_with_score": sum(1 for row in arm_covid if row.get("covid_score") is not None),
        "n_with_logL": sum(1 for row in arm_covid
                           if (row.get("extra") or {}).get("candidate_log_likelihoods")),
        "n_with_decision": sum(1 for row in arm_covid if row.get("covid_pred") is not None),
        "n_scoring_errors": sum(1 for row in arm_covid
                                if (row.get("extra") or {}).get("scoring_error")),
        # Direct evidence of an unbound image, read off the saved raw outputs.
        "image_absent_rate": (
            round(sum(1 for row in predictions if row["arm"] == arm
                      and any(phrase in (row.get("raw_output") or "").lower()
                              for phrase in IMAGE_ABSENT_PHRASES))
                  / max(sum(1 for row in predictions if row["arm"] == arm), 1), 4)),
        "in_validation_table": arm in {row["arm"] for row in validation_rows},
    })
availability = pd.DataFrame(availability_rows)
availability.to_csv(NB07_DIR / "score_availability.csv", index=False)
print(f"Score availability (validation source: {validation_source}):")
print(availability.to_string(index=False))
for stats in availability_rows:
    rate = stats.get("image_absent_rate")
    if rate is not None and rate > 0.5:
        print()
        print(f"  {stats['arm']}: {rate:.1%} of raw outputs match refusal phrasing "
              "(\"please provide...\", \"cannot see...\").")
        if stats["arm"] in KNOWN_INCOMPATIBLE_OUTPUT_FORMAT:
            # Section 9b established that the image IS bound for this arm and the model does
            # read it. The refusal is a response to the schema-only prompt, not evidence of an
            # unbound image. Do not restate the earlier, disproven explanation here.
            print("  NOT an image-binding failure: section 9b confirms the image is bound and "
                  "the model")
            print("  describes the radiograph when asked to reason. Refusal is a response to "
                  "the")
            print("  schema-only prompt (~98% refusal) versus a reasoning prompt (~54%), and it "
                  "rises")
            print("  with severity. See nvreason_finding.json; report it, do not 'fix' it.")
        else:
            print("  Check whether the image reached the model at all: run section 6b, and if "
                  "it")
            print(f"  reports a binding failure, regenerate with RESET_ARMS=['{stats['arm']}'].")
print()

expected_internal = {str(key) for key in work[~work["is_external"]]["image_key"]}
for arm in RUN_ARMS:
    for task in TASKS:
        covered = {row["image_key"] for row in rows_for(arm, task=task)}
        missing = expected_internal - covered
        if missing:
            failures.append(f"{arm}/{task}: {len(missing)} internal images have no prediction "
                            f"(e.g. {sorted(missing)[:3]}).")

# ---- Protocol 7.2: quarantine, do not abort ----------------------------------------------
# An arm whose scored decision does not reproduce its generated decision has an unusable
# continuous score. Aborting the notebook would discard the other arms' ~20 hours of work and
# fix nothing. Instead the arm is marked score_usable=False, which NB 17/18 must honour, and
# the failure is recorded loudly. The gate still fails if EVERY arm is unusable, because then
# there is no AUROC anywhere and referee 1.3 is left unanswered.
usability = defaultdict(lambda: {"score_usable": None, "mrale_usable": None})
score_usability = {}
quarantined = []
mrale_quarantined = []
for row in validation_rows:
    usable = bool(row["meets_protocol_7_2"])
    score_usability[row["arm"]] = {
        "score_usable": usable,
        "agreement": row["score_generation_agreement"],
        "threshold": SCORE_AGREEMENT_THRESHOLD,
        "decisive_agreement": row.get("decisive_agreement"),
        "near_tie_fraction": row.get("near_tie_fraction"),
        "gated_on": "decisive" if GATE_ON_DECISIVE_AGREEMENT else "overall",
        "aligned": row["arm"] in RESCORE_ARMS,
    }
    if usable:
        continue
    quarantined.append(row["arm"])
    message = (
        f"{row['arm']}: overall agreement {row['score_generation_agreement']:.4f}, "
        f"decisive-case {row.get('decisive_agreement')} "
        f"(margin >= {AGREEMENT_DECISIVE_MARGIN} nats, near-tie fraction "
        f"{row.get('near_tie_fraction')}) below {SCORE_AGREEMENT_THRESHOLD}. Its "
        "continuous score is NOT a faithful description of the model's own decisions, so its "
        "AUROC/AUPRC/DeLong results are invalid and it is marked score_usable=False for "
        "NB 17/18. Hard-decision metrics (accuracy, sensitivity, specificity, F1, MCC) remain "
        "valid for this arm. Try section 8b: add it to RESCORE_ARMS to score in the model's "
        "own emitted format."
    )
    if QUARANTINE_FAILING_SCORE_ARMS:
        warnings.append(message)
    else:
        failures.append(message)

for arm, entry in score_usability.items():
    usability[arm]["score_usable"] = entry["score_usable"]
    usability[arm]["score_agreement"] = entry["agreement"]

# The previous rule compared `quarantined` against `len(validation_rows)`, so it fired when
# 2 of 3 arms failed and the third was simply ABSENT from the table -- reporting "every arm
# failed" when the truth was "two failed and one was never scored". Those need different fixes,
# so they are now separate checks, and only a genuine total absence of usable scores blocks.
arms_in_table = {row["arm"] for row in validation_rows}
arms_absent = [arm for arm in RUN_ARMS if arm not in arms_in_table]
arms_usable = sorted(arms_in_table - set(quarantined))

for arm in arms_absent:
    row = next((item for item in availability_rows if item["arm"] == arm), {})
    if arm in KNOWN_INCOMPATIBLE_OUTPUT_FORMAT:
        warnings.append(
            f"{arm}: output format incompatible with the requested schema "
            f"({KNOWN_INCOMPATIBLE_OUTPUT_FORMAT[arm]}). The image IS bound and the model does "
            "read it (section 9b). No AUROC and no mRALE are available under zero-shot, but "
            "that is a property of the model's output space, not a pipeline defect. Report it; "
            "do not report 24.0 or a chance AUROC as if they measured ability."
        )
    elif row.get("image_absent_rate") is not None and row["image_absent_rate"] > 0.5:
        warnings.append(
            f"{arm}: {row['image_absent_rate']:.1%} of raw outputs match refusal phrasing. "
            "Run section 6b to establish whether the image is reaching the model; if it is "
            f"not, regenerate with RESET_ARMS=['{arm}']."
        )
    elif row.get("n_with_score", 0) == 0:
        warnings.append(
            f"{arm}: no continuous scores at all ({row.get('n_covid_rows', 0)} COVID rows, "
            f"{row.get('n_scoring_errors', 0)} scoring errors). Absent from the validation "
            "table, so no AUROC and implicitly score_usable=False. Check "
            f"extra.scoring_error; if scoring never ran, RESET_ARMS=['{arm}'] will produce it."
        )
    else:
        warnings.append(
            f"{arm}: has {row.get('n_with_score')} scores but no validation row, which means "
            "its generated decisions are all null. Its JSON is not parsing -- see "
            "output_integrity.csv."
        )
    usability[arm]["score_usable"] = False

if RUN_TOKEN_SCORING and not arms_usable:
    failures.append(
        "No arm has a usable continuous score: "
        f"quarantined={quarantined or 'none'}, absent={arms_absent or 'none'}. Fig. 3 (ROC/PR) "
        "cannot be produced and referee 1.3 stays unanswered. Because this holds across every "
        "arm it is a systematic scoring problem, not a per-model quirk -- most often the chat "
        "template inserting tokens between the prompt and the answer, or candidate strings "
        "that are not token-identical to what the models emit. Read "
        "score_disagreement_diagnosis.csv, then repair with RESCORE_ARMS "
        f"({sorted(arms_in_table)}) before changing anything else."
    )
elif quarantined:
    warnings.append(
        f"Usable continuous scores remain for {arms_usable}; quarantined {quarantined}. "
        "Fig. 3 can still be produced from the usable arms, with the quarantined ones omitted "
        "and the omission stated in the caption."
    )

if RUN_TOKEN_SCORING and not validation_rows:
    scored_rows = sum(1 for row in predictions
                      if row.get("task") == "covid_classification"
                      and row.get("covid_score") is not None)
    if scored_rows:
        failures.append(
            f"No validated scores, but {scored_rows:,} prediction rows DO carry a covid_score. "
            "The backfill in section 7 should have recovered them into score_records.jsonl -- "
            "re-run section 7 (it will not regenerate anything; the per-item cache is intact) "
            "and then section 8."
        )
    else:
        failures.append(
            "Token scoring was enabled but produced no scores at all. Without it there is no "
            "AUROC for any generative arm, which is the core of referee 1.3 and 2d. Check "
            "extra.scoring_error on the prediction rows."
        )

for arm in RUN_ARMS:
    merged = merge_tasks(arm)
    if not merged:
        continue
    metrics = evaluate_arm(merged, arm)
    coverage = metrics.get("mrale", {}).get("coverage", float("nan"))
    if not math.isnan(coverage) and coverage <= MIN_ACCEPTABLE_MRALE_COVERAGE:
        # Exactly zero parseable outputs is a format bug, not a weak model, and a 24.0 MAE for
        # it would be actively misleading. Quarantine the arm's mRALE output rather than
        # blocking the notebook -- same policy as the score quarantine above.
        usability[arm]["mrale_usable"] = False
        mrale_quarantined.append(arm)
        message = (
            f"{arm}: mRALE coverage is {coverage:.3f} -- NOT ONE output parsed. That is a "
            "format mismatch, not a result: the penalised MAE is exactly the 24-point invalid "
            "penalty and says nothing about the model. Marked mrale_usable=False, so this "
            "arm's mRALE row must be OMITTED from Table 2 rather than printed as 24.0. See the "
            "section 9 triage for raw outputs; the usual fix for a chain-of-thought model is "
            f"ARM_PROMPT_SUFFIX['{arm}'] plus RESET_ARMS=['{arm}'], which is a disclosable "
            "deviation affecting RQ2 comparability."
        )
        if QUARANTINE_ZERO_COVERAGE_ARMS:
            warnings.append(message)
        else:
            failures.append(message)
    elif not math.isnan(coverage) and coverage < 0.5:
        warnings.append(
            f"{arm}: mRALE coverage only {coverage:.3f}. The penalised MAE (endpoint P1) is "
            "dominated by the 24-point invalid penalty rather than by prediction error. Report "
            "coverage beside every MAE for this arm."
        )
    if usability[arm]["mrale_usable"] is None:
        usability[arm]["mrale_usable"] = True
    usability[arm]["mrale_coverage"] = (None if math.isnan(coverage) else round(coverage, 4))

    auroc = metrics.get("covid", {}).get("auroc", float("nan"))
    if not math.isnan(auroc) and auroc < 0.55:
        warnings.append(f"{arm}: AUROC {auroc:.4f} is near chance zero-shot. Expected for an "
                        "unadapted model on PCR status; it is the baseline the LoRA arms must "
                        "beat.")

if mrale_quarantined and len(mrale_quarantined) == len(RUN_ARMS):
    failures.append(
        f"EVERY arm produced zero parseable mRALE output ({mrale_quarantined}). That is a "
        "systematic prompt or parsing bug, not three independent model failures: endpoint P1 "
        "cannot be computed for any generative arm. Inspect the section 9 raw outputs before "
        "changing anything else."
    )

cm.write_json(NB07_DIR / "usability.json", dict(usability))

# Stamp the health flags onto arm_summary.csv itself. usability.json is authoritative, but a
# reader who opens only the summary must not be able to lift a 24.0 or a chance AUROC into
# Table 2 without seeing that the arm is quarantined.
summary_path = NB07_DIR / "arm_summary.csv"
if summary_path.is_file():
    stamped = pd.read_csv(summary_path)
    stamped["score_usable"] = stamped["arm"].map(
        lambda a: usability.get(a, {}).get("score_usable"))
    stamped["mrale_usable"] = stamped["arm"].map(
        lambda a: usability.get(a, {}).get("mrale_usable"))
    stamped["report_in_table2"] = stamped["arm"].map(
        lambda a: "omit mRALE; omit AUROC" if not (
            usability.get(a, {}).get("mrale_usable") or usability.get(a, {}).get("score_usable"))
        else ("omit AUROC (score quarantined)"
              if not usability.get(a, {}).get("score_usable")
              else ("omit mRALE (coverage 0)"
                    if not usability.get(a, {}).get("mrale_usable") else "full row")))
    stamped.to_csv(summary_path, index=False)
    print()
    print("arm_summary.csv stamped with usability flags:")
    print(stamped[["arm", "score_usable", "mrale_usable", "report_in_table2"]]
          .to_string(index=False))
cm.write_json(NB07_DIR / "score_usability.json", score_usability)

if ("duplicate_object_rate" in getattr(integrity, "columns", [])
        and len(integrity) and integrity["duplicate_object_rate"].notna().any()
        and integrity["duplicate_object_rate"].max() > 0.01):
    warnings.append(
        f"Max duplicate-object rate {integrity['duplicate_object_rate'].max():.3f}. The E6-Q "
        "stop criteria are not firing everywhere; fix before running the E6-C "
        "self-consistency arm."
    )

if MAX_IMAGES_PER_ARM is not None:
    warnings.append(f"WIRING CHECK MODE: only {MAX_IMAGES_PER_ARM} images per arm. Set "
                    "MAX_IMAGES_PER_ARM=None before quoting any number.")


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

cm.write_json(NB07_DIR / "gate_nb07.json",
              {"passed": not failures, "failures": failures, "warnings": warnings,
               "score_validation": validation_rows,
               "score_usability": score_usability,
               "usability": dict(usability),
               "validation_source": validation_source,
               "score_availability": availability_rows,
               "arms_usable_for_auroc": arms_usable,
               "arms_absent_from_validation": arms_absent,
               "quarantined_score_arms": quarantined,
               "quarantined_mrale_arms": mrale_quarantined,
               "stage_status": _load_status()})
if quarantined or mrale_quarantined:
    print()
    print("QUARANTINE SUMMARY (see usability.json; NB 17-18 must honour it)")
    for arm, entry in sorted(usability.items()):
        print(f"  {arm:<16} score_usable={entry['score_usable']} "
              f"mrale_usable={entry['mrale_usable']} "
              f"coverage={entry.get('mrale_coverage')}")
    if quarantined:
        print(f"  score_usable=False -> {quarantined}: excluded from ROC/PR/DeLong/"
              "calibration. Hard-decision metrics remain valid and stay in Table 2.")
    if mrale_quarantined:
        print(f"  mrale_usable=False -> {mrale_quarantined}: OMIT the mRALE row from Table 2. "
              "Printing 24.0 would misreport a parsing failure as a severity result.")
# Reasons are embedded in the assertion message, so a pasted traceback is
# self-explanatory without the printed output above.
if failures:
    detail = "\n".join(f"  [{index + 1}] {message}"
                       for index, message in enumerate(failures))
    raise AssertionError(
        f"NB 07 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 07 gate: PASSED")
print("Stage B baseline suite (E0a-E0g) is complete once NB 05, 06, and 07 have all run.")

## Notes carried forward

- **`token_probability_scores.parquet` is the artifact that unblocks the statistics plan.**
  NB 17 (statistics/DeLong) and NB 18 (ROC/PR curves and calibration) both depend on it. Without it,
  the revision cannot answer referee 1.3, so treat a scoring failure as blocking rather than
  cosmetic.
- The same scoring functions must be reused verbatim in NB 09/10 for the LoRA arms. A
  zero-shot AUROC computed by sequence likelihood is not comparable to a LoRA AUROC computed
  some other way.
- `mrale_total` is always the constrained `right + left`. The model's own `mRALE Score` field
  is preserved in `extra.reported_total`, and `formula_consistency` measures how often the two
  agree — a genuine interpretability signal, since a model that cannot add its own components
  is not reasoning about them.
- `nvreason_traces.jsonl` feeds E10c (evidence grounding, unsupported-claim rate). It is the
  only arm producing free-form reasoning at this stage, so it is the natural pilot for the
  grounding rubric before the full reasoner in NB 15 and the grounding audit in NB 20.
- Zero-shot latency in `output_integrity.csv` goes into Table 10 and is the honest counterweight
  to the framework's accuracy claims.